# Évaluation du modèle Camembert NER


Ce notebook évalue les performances du modèle `CamembertNERModel`, qui extrait les villes de départ (B-DEP, I-DEP)
et de destination (B-ARR, I-ARR) à partir des textes d'entrée. Les performances du modèle sont analysées en fonction
des métriques classiques et d'autres visualisations telles que la distribution des scores de confiance et des longueurs de texte.


## Aperçu des données de test

In [ ]:

from datasets import load_dataset

# Load dataset
dataset = load_dataset("csv", data_files={"test": r"C:\Users\leogu\Desktop\Projects\nlp_travel_order_resolver\datasets/camembert_ner_dataset.csv"})["test"]
dataset.to_pandas().head()


## Résultats des prédictions au niveau des tokens

In [ ]:

import pandas as pd
import numpy as np

# Display predictions
results = [{'text': "Elle cherche un itinéraire jusqu'à Nice à Paris", 'tokens': ['<s>', '▁Elle', '▁cherche', '▁un', '▁itinéraire', '▁jusqu', "'", 'à', '▁Nice', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je veux aller de Biarritz à Marseille', 'tokens': ['<s>', '▁Je', '▁veux', '▁aller', '▁de', '▁Biarritz', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de partir de Lyon à Toulouse demain', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁partir', '▁de', '▁Lyon', '▁à', '▁Toulouse', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0], dtype=int64)}, {'text': 'Nous réservons un billet pour un trajet de Bordeaux à Nantes', 'tokens': ['<s>', '▁Nous', '▁réserv', 'ons', '▁un', '▁billet', '▁pour', '▁un', '▁trajet', '▁de', '▁Bordeaux', '▁à', '▁Nantes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle va prendre le train de Strasbourg à Lille', 'tokens': ['<s>', '▁Elle', '▁va', '▁prendre', '▁le', '▁train', '▁de', '▁Strasbourg', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il part de Montpellier pour aller à Lyon', 'tokens': ['<s>', '▁Il', '▁part', '▁de', '▁Montpellier', '▁pour', '▁aller', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle cherche un vol de Paris à Nice', 'tokens': ['<s>', '▁Elle', '▁cherche', '▁un', '▁vol', '▁de', '▁Paris', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': "Il prend l'avion pour un trajet entre Lyon et Marseille", 'tokens': ['<s>', '▁Il', '▁prend', '▁l', "'", 'avion', '▁pour', '▁un', '▁trajet', '▁entre', '▁Lyon', '▁et', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de partir de Toulouse pour visiter Bordeaux', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁partir', '▁de', '▁Toulouse', '▁pour', '▁visiter', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un itinéraire en train de Nantes à Rennes', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁itinéraire', '▁en', '▁train', '▁de', '▁Nantes', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous devons partir demain matin de Marseille à Strasbourg', 'tokens': ['<s>', '▁Nous', '▁devons', '▁partir', '▁demain', '▁matin', '▁de', '▁Marseille', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je dois rejoindre mes amis de Bordeaux à Paris', 'tokens': ['<s>', '▁Je', '▁dois', '▁rejoindre', '▁mes', '▁amis', '▁de', '▁Bordeaux', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il a prévu de voyager de Lille à Toulouse', 'tokens': ['<s>', '▁Il', '▁a', '▁prévu', '▁de', '▁voyager', '▁de', '▁Lille', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je pars de Nice pour un séjour à Paris', 'tokens': ['<s>', '▁Je', '▁par', 's', '▁de', '▁Nice', '▁pour', '▁un', '▁séjour', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Rennes à Lyon', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Rennes', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Ils veulent un trajet de Marseille à Bordeaux', 'tokens': ['<s>', '▁Ils', '▁veulent', '▁un', '▁trajet', '▁de', '▁Marseille', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle se rendra de Strasbourg à Lille en train', 'tokens': ['<s>', '▁Elle', '▁se', '▁rendra', '▁de', '▁Strasbourg', '▁à', '▁Lille', '▁en', '▁train', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Nous allons en voiture de Toulouse à Montpellier', 'tokens': ['<s>', '▁Nous', '▁allons', '▁en', '▁voiture', '▁de', '▁Toulouse', '▁à', '▁Montpellier', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il cherche un vol de Bordeaux à Paris', 'tokens': ['<s>', '▁Il', '▁cherche', '▁un', '▁vol', '▁de', '▁Bordeaux', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle a prévu un trajet de Marseille à Nantes', 'tokens': ['<s>', '▁Elle', '▁a', '▁prévu', '▁un', '▁trajet', '▁de', '▁Marseille', '▁à', '▁Nantes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Ils prennent un train de Rennes à Lyon ce soir', 'tokens': ['<s>', '▁Ils', '▁prennent', '▁un', '▁train', '▁de', '▁Rennes', '▁à', '▁Lyon', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Nous partons demain matin de Paris à Lille', 'tokens': ['<s>', '▁Nous', '▁partons', '▁demain', '▁matin', '▁de', '▁Paris', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il a acheté un billet de Toulouse à Bordeaux', 'tokens': ['<s>', '▁Il', '▁a', '▁acheté', '▁un', '▁billet', '▁de', '▁Toulouse', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': "Elle prend l'avion de Nice à Lyon la semaine prochaine", 'tokens': ['<s>', '▁Elle', '▁prend', '▁l', "'", 'avion', '▁de', '▁Nice', '▁à', '▁Lyon', '▁la', '▁semaine', '▁prochaine', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0], dtype=int64)}, {'text': 'Nous cherchons un trajet de Nantes à Paris', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁trajet', '▁de', '▁Nantes', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il voyage de Lille à Marseille pour un événement', 'tokens': ['<s>', '▁Il', '▁voyage', '▁de', '▁Lille', '▁à', '▁Marseille', '▁pour', '▁un', '▁événement', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0], dtype=int64)}, {'text': 'Elle prend le train de Rennes à Strasbourg', 'tokens': ['<s>', '▁Elle', '▁prend', '▁le', '▁train', '▁de', '▁Rennes', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je prévois de partir de Paris pour aller à Lyon', 'tokens': ['<s>', '▁Je', '▁pré', 'vois', '▁de', '▁partir', '▁de', '▁Paris', '▁pour', '▁aller', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un itinéraire de Toulouse à Marseille', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁itinéraire', '▁de', '▁Toulouse', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit un trajet de Lille à Nice', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Lille', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il part en train de Rennes à Paris', 'tokens': ['<s>', '▁Il', '▁part', '▁en', '▁train', '▁de', '▁Rennes', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous allons voyager de Marseille à Toulouse', 'tokens': ['<s>', '▁Nous', '▁allons', '▁voyager', '▁de', '▁Marseille', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un billet de train de Bordeaux à Nantes', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁billet', '▁de', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Nantes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle a réservé un vol de Paris à Marseille', 'tokens': ['<s>', '▁Elle', '▁a', '▁réservé', '▁un', '▁vol', '▁de', '▁Paris', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il doit rejoindre un client de Strasbourg à Lyon', 'tokens': ['<s>', '▁Il', '▁doit', '▁rejoindre', '▁un', '▁client', '▁de', '▁Strasbourg', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons prévu de partir de Lille à Rennes', 'tokens': ['<s>', '▁Nous', '▁avons', '▁prévu', '▁de', '▁partir', '▁de', '▁Lille', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je dois prendre un avion de Marseille à Paris', 'tokens': ['<s>', '▁Je', '▁dois', '▁prendre', '▁un', '▁avion', '▁de', '▁Marseille', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle part de Lyon pour visiter Toulouse', 'tokens': ['<s>', '▁Elle', '▁part', '▁de', '▁Lyon', '▁pour', '▁visiter', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un itinéraire de train de Bordeaux à Nice', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁itinéraire', '▁de', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de partir de Paris à Rennes', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁partir', '▁de', '▁Paris', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut prendre un train de Lille à Marseille', 'tokens': ['<s>', '▁Elle', '▁veut', '▁prendre', '▁un', '▁train', '▁de', '▁Lille', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons un voyage de Rennes à Paris', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁un', '▁voyage', '▁de', '▁Rennes', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je vais voyager de Toulouse à Lille', 'tokens': ['<s>', '▁Je', '▁vais', '▁voyager', '▁de', '▁Toulouse', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle doit rejoindre des collègues de Paris à Lyon', 'tokens': ['<s>', '▁Elle', '▁doit', '▁rejoindre', '▁des', '▁collègues', '▁de', '▁Paris', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un trajet en train de Bordeaux à Marseille', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁trajet', '▁en', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prend un avion de Nice à Strasbourg', 'tokens': ['<s>', '▁Il', '▁prend', '▁un', '▁avion', '▁de', '▁Nice', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle part de Rennes pour visiter Nantes', 'tokens': ['<s>', '▁Elle', '▁part', '▁de', '▁Rennes', '▁pour', '▁visiter', '▁Nantes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous réservons un billet de Paris à Toulouse', 'tokens': ['<s>', '▁Nous', '▁réserv', 'ons', '▁un', '▁billet', '▁de', '▁Paris', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit un vol de Lyon à Lille', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁un', '▁vol', '▁de', '▁Lyon', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un train de Bordeaux à Marseille', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Toulouse à Rennes', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Toulouse', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je prévois de prendre un train de Paris à Nice', 'tokens': ['<s>', '▁Je', '▁pré', 'vois', '▁de', '▁prendre', '▁un', '▁train', '▁de', '▁Paris', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons demain de Lille à Bordeaux', 'tokens': ['<s>', '▁Nous', '▁partons', '▁demain', '▁de', '▁Lille', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut réserver un billet de Rennes à Lyon', 'tokens': ['<s>', '▁Elle', '▁veut', '▁réserver', '▁un', '▁billet', '▁de', '▁Rennes', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je dois aller de Paris à Nantes', 'tokens': ['<s>', '▁Je', '▁dois', '▁aller', '▁de', '▁Paris', '▁à', '▁Nantes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit un voyage de Marseille à Lille', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁un', '▁voyage', '▁de', '▁Marseille', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons pris un billet de Toulouse à Bordeaux', 'tokens': ['<s>', '▁Nous', '▁avons', '▁pris', '▁un', '▁billet', '▁de', '▁Toulouse', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il part en voiture de Lille à Paris', 'tokens': ['<s>', '▁Il', '▁part', '▁en', '▁voiture', '▁de', '▁Lille', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je prends un vol de Marseille à Strasbourg', 'tokens': ['<s>', '▁Je', '▁prends', '▁un', '▁vol', '▁de', '▁Marseille', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle part en train de Bordeaux à Rennes', 'tokens': ['<s>', '▁Elle', '▁part', '▁en', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous réservons un billet de Lyon à Marseille', 'tokens': ['<s>', '▁Nous', '▁réserv', 'ons', '▁un', '▁billet', '▁de', '▁Lyon', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Toulouse à Paris', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Toulouse', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un itinéraire de train de Rennes à Lille', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁itinéraire', '▁de', '▁train', '▁de', '▁Rennes', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prend un avion de Marseille à Nantes', 'tokens': ['<s>', '▁Il', '▁prend', '▁un', '▁avion', '▁de', '▁Marseille', '▁à', '▁Nantes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de partir de Paris à Rennes', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁partir', '▁de', '▁Paris', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous allons de Lyon à Bordeaux demain', 'tokens': ['<s>', '▁Nous', '▁allons', '▁de', '▁Lyon', '▁à', '▁Bordeaux', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 2, 0, 0], dtype=int64)}, {'text': 'Il prend un train de Lille à Toulouse', 'tokens': ['<s>', '▁Il', '▁prend', '▁un', '▁train', '▁de', '▁Lille', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle cherche un trajet de Rennes à Marseille', 'tokens': ['<s>', '▁Elle', '▁cherche', '▁un', '▁trajet', '▁de', '▁Rennes', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons un voyage de Bordeaux à Lyon', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁un', '▁voyage', '▁de', '▁Bordeaux', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je dois aller de Paris à Marseille demain', 'tokens': ['<s>', '▁Je', '▁dois', '▁aller', '▁de', '▁Paris', '▁à', '▁Marseille', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0, 0], dtype=int64)}, {'text': 'Elle prévoit de rejoindre ses amis de Lille à Toulouse', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁rejoindre', '▁ses', '▁amis', '▁de', '▁Lille', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il part de Rennes pour un événement à Paris', 'tokens': ['<s>', '▁Il', '▁part', '▁de', '▁Rennes', '▁pour', '▁un', '▁événement', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous réservons un billet de Marseille à Lille', 'tokens': ['<s>', '▁Nous', '▁réserv', 'ons', '▁un', '▁billet', '▁de', '▁Marseille', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion de Toulouse à Lyon', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁de', '▁Toulouse', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je pars de Paris pour visiter Rennes', 'tokens': ['<s>', '▁Je', '▁par', 's', '▁de', '▁Paris', '▁pour', '▁visiter', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un trajet de train de Lille à Bordeaux', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁trajet', '▁de', '▁train', '▁de', '▁Lille', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il voyage de Rennes à Nice', 'tokens': ['<s>', '▁Il', '▁voyage', '▁de', '▁Rennes', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un billet de train de Paris à Marseille', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁billet', '▁de', '▁train', '▁de', '▁Paris', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Bordeaux à Lyon demain', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Bordeaux', '▁à', '▁Lyon', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0], dtype=int64)}, {'text': 'Nous avons un vol prévu de Toulouse à Nice', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁vol', '▁prévu', '▁de', '▁Toulouse', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prend le bus de Rennes à Strasbourg ce soir', 'tokens': ['<s>', '▁Il', '▁prend', '▁le', '▁bus', '▁de', '▁Rennes', '▁à', '▁Strasbourg', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Elle veut un itinéraire de train de Lille à Paris', 'tokens': ['<s>', '▁Elle', '▁veut', '▁un', '▁itinéraire', '▁de', '▁train', '▁de', '▁Lille', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je dois partir de Lyon pour une réunion à Marseille', 'tokens': ['<s>', '▁Je', '▁dois', '▁partir', '▁de', '▁Lyon', '▁pour', '▁une', '▁réunion', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un billet de train pour un voyage de Nantes à Paris', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁billet', '▁de', '▁train', '▁pour', '▁un', '▁voyage', '▁de', '▁Nantes', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de prendre le train de Marseille à Lyon', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁prendre', '▁le', '▁train', '▁de', '▁Marseille', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un itinéraire de Bordeaux à Nice', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁itinéraire', '▁de', '▁Bordeaux', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de partir de Rennes pour visiter Lille', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁partir', '▁de', '▁Rennes', '▁pour', '▁visiter', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion de Toulouse à Strasbourg', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁de', '▁Toulouse', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous allons voyager de Lyon à Bordeaux demain', 'tokens': ['<s>', '▁Nous', '▁allons', '▁voyager', '▁de', '▁Lyon', '▁à', '▁Bordeaux', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0, 0], dtype=int64)}, {'text': 'Il part en train de Paris à Nice', 'tokens': ['<s>', '▁Il', '▁part', '▁en', '▁train', '▁de', '▁Paris', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut réserver un billet de Rennes à Marseille', 'tokens': ['<s>', '▁Elle', '▁veut', '▁réserver', '▁un', '▁billet', '▁de', '▁Rennes', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je prévois de prendre le train de Nantes à Lyon', 'tokens': ['<s>', '▁Je', '▁pré', 'vois', '▁de', '▁prendre', '▁le', '▁train', '▁de', '▁Nantes', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un vol prévu de Lille à Toulouse', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁vol', '▁prévu', '▁de', '▁Lille', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle doit rejoindre des amis de Marseille à Paris', 'tokens': ['<s>', '▁Elle', '▁doit', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Marseille', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de voyager de Bordeaux à Strasbourg', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Bordeaux', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un itinéraire de Rennes à Toulouse', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁itinéraire', '▁de', '▁Rennes', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet pour un trajet de Nice à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁pour', '▁un', '▁trajet', '▁de', '▁Nice', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je veux partir de Lyon pour un événement à Paris', 'tokens': ['<s>', '▁Je', '▁veux', '▁partir', '▁de', '▁Lyon', '▁pour', '▁un', '▁événement', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Il partira demain de Toulouse à Lille', 'tokens': ['<s>', '▁Il', '▁partir', 'a', '▁demain', '▁de', '▁Toulouse', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un train de Rennes à Lyon', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁train', '▁de', '▁Rennes', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous réservons un vol de Marseille à Nice', 'tokens': ['<s>', '▁Nous', '▁réserv', 'ons', '▁un', '▁vol', '▁de', '▁Marseille', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il veut voyager de Paris à Strasbourg', 'tokens': ['<s>', '▁Il', '▁veut', '▁voyager', '▁de', '▁Paris', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit un trajet de Bordeaux à Toulouse', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Bordeaux', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un billet pour un voyage de Lille à Lyon', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁billet', '▁pour', '▁un', '▁voyage', '▁de', '▁Lille', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre ses collègues de Rennes à Paris', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁ses', '▁collègues', '▁de', '▁Rennes', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je dois partir de Marseille pour un séjour à Bordeaux', 'tokens': ['<s>', '▁Je', '▁dois', '▁partir', '▁de', '▁Marseille', '▁pour', '▁un', '▁séjour', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Il prend le bus de Nice à Toulouse ce soir', 'tokens': ['<s>', '▁Il', '▁prend', '▁le', '▁bus', '▁de', '▁Nice', '▁à', '▁Toulouse', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Lyon à Rennes demain', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Lyon', '▁à', '▁Rennes', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0], dtype=int64)}, {'text': 'Nous réservons un vol de Paris à Lille', 'tokens': ['<s>', '▁Nous', '▁réserv', 'ons', '▁un', '▁vol', '▁de', '▁Paris', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle partira en train de Bordeaux à Marseille', 'tokens': ['<s>', '▁Elle', '▁partir', 'a', '▁en', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de rejoindre des amis de Rennes à Nice', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Rennes', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un trajet prévu de Toulouse à Lyon', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁trajet', '▁prévu', '▁de', '▁Toulouse', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle doit partir de Paris pour un événement à Marseille', 'tokens': ['<s>', '▁Elle', '▁doit', '▁partir', '▁de', '▁Paris', '▁pour', '▁un', '▁événement', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un itinéraire de Rennes à Bordeaux', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁itinéraire', '▁de', '▁Rennes', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il part de Lille pour un séjour à Strasbourg', 'tokens': ['<s>', '▁Il', '▁part', '▁de', '▁Lille', '▁pour', '▁un', '▁séjour', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion de Nice à Lyon la semaine prochaine', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁de', '▁Nice', '▁à', '▁Lyon', '▁la', '▁semaine', '▁prochaine', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0], dtype=int64)}, {'text': 'Nous prévoyons de prendre le train de Bordeaux à Lille', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁prendre', '▁le', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre des amis de Paris à Toulouse', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Paris', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Marseille à Rennes', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Marseille', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de partir de Lyon pour visiter Lille', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁partir', '▁de', '▁Lyon', '▁pour', '▁visiter', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je pars de Rennes pour un voyage à Paris', 'tokens': ['<s>', '▁Je', '▁par', 's', '▁de', '▁Rennes', '▁pour', '▁un', '▁voyage', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet de train de Toulouse à Nice', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁de', '▁train', '▁de', '▁Toulouse', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous réservons un vol de Bordeaux à Strasbourg', 'tokens': ['<s>', '▁Nous', '▁réserv', 'ons', '▁un', '▁vol', '▁de', '▁Bordeaux', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il cherche un itinéraire de Paris à Lyon', 'tokens': ['<s>', '▁Il', '▁cherche', '▁un', '▁itinéraire', '▁de', '▁Paris', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de rejoindre ses collègues de Lille à Marseille', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁rejoindre', '▁ses', '▁collègues', '▁de', '▁Lille', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous devons partir de Nice pour un événement à Rennes', 'tokens': ['<s>', '▁Nous', '▁devons', '▁partir', '▁de', '▁Nice', '▁pour', '▁un', '▁événement', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un vol de Toulouse à Paris', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁vol', '▁de', '▁Toulouse', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle a prévu de voyager de Bordeaux à Marseille', 'tokens': ['<s>', '▁Elle', '▁a', '▁prévu', '▁de', '▁voyager', '▁de', '▁Bordeaux', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un billet pour un trajet de Lyon à Strasbourg', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁billet', '▁pour', '▁un', '▁trajet', '▁de', '▁Lyon', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut partir de Lille pour visiter Rennes', 'tokens': ['<s>', '▁Elle', '▁veut', '▁partir', '▁de', '▁Lille', '▁pour', '▁visiter', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je prévois de prendre le train de Marseille à Paris', 'tokens': ['<s>', '▁Je', '▁pré', 'vois', '▁de', '▁prendre', '▁le', '▁train', '▁de', '▁Marseille', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prend un bus de Bordeaux à Nice ce soir', 'tokens': ['<s>', '▁Il', '▁prend', '▁un', '▁bus', '▁de', '▁Bordeaux', '▁à', '▁Nice', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Elle prévoit un itinéraire de Rennes à Toulouse', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁un', '▁itinéraire', '▁de', '▁Rennes', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous allons partir de Lyon pour un événement à Lille', 'tokens': ['<s>', '▁Nous', '▁allons', '▁partir', '▁de', '▁Lyon', '▁pour', '▁un', '▁événement', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut prendre un billet de train de Nice à Paris', 'tokens': ['<s>', '▁Elle', '▁veut', '▁prendre', '▁un', '▁billet', '▁de', '▁train', '▁de', '▁Nice', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de rejoindre des amis de Marseille à Rennes', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Marseille', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un trajet de Toulouse à Lyon', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁trajet', '▁de', '▁Toulouse', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion de Bordeaux à Paris', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁de', '▁Bordeaux', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons en train de Lille à Nice', 'tokens': ['<s>', '▁Nous', '▁partons', '▁en', '▁train', '▁de', '▁Lille', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de voyager de Rennes à Strasbourg', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Rennes', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut prendre un vol de Lyon à Toulouse', 'tokens': ['<s>', '▁Elle', '▁veut', '▁prendre', '▁un', '▁vol', '▁de', '▁Lyon', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un billet de train de Marseille à Lille', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁billet', '▁de', '▁train', '▁de', '▁Marseille', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un vol de Paris à Bordeaux', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁vol', '▁de', '▁Paris', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle partira de Nice pour un événement à Rennes', 'tokens': ['<s>', '▁Elle', '▁partir', 'a', '▁de', '▁Nice', '▁pour', '▁un', '▁événement', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit un voyage de Toulouse à Marseille', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁un', '▁voyage', '▁de', '▁Toulouse', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un trajet de Lille à Paris prévu', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁trajet', '▁de', '▁Lille', '▁à', '▁Paris', '▁prévu', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0], dtype=int64)}, {'text': 'Elle veut rejoindre ses amis de Rennes à Lyon', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁ses', '▁amis', '▁de', '▁Rennes', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je pars de Bordeaux pour visiter Marseille', 'tokens': ['<s>', '▁Je', '▁par', 's', '▁de', '▁Bordeaux', '▁pour', '▁visiter', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de prendre un bus de Lyon à Nice', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁prendre', '▁un', '▁bus', '▁de', '▁Lyon', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle a réservé un billet pour un trajet de Paris à Toulouse', 'tokens': ['<s>', '▁Elle', '▁a', '▁réservé', '▁un', '▁billet', '▁pour', '▁un', '▁trajet', '▁de', '▁Paris', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Marseille à Strasbourg', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Marseille', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Lille à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Lille', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un billet de train de Rennes à Marseille', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁billet', '▁de', '▁train', '▁de', '▁Rennes', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de prendre un vol de Toulouse à Paris', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁prendre', '▁un', '▁vol', '▁de', '▁Toulouse', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit un trajet de Nice à Lyon', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Nice', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il part de Bordeaux pour un séjour à Lille', 'tokens': ['<s>', '▁Il', '▁part', '▁de', '▁Bordeaux', '▁pour', '▁un', '▁séjour', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet pour un trajet de Rennes à Nice', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁pour', '▁un', '▁trajet', '▁de', '▁Rennes', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un vol de Paris à Lyon', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁vol', '▁de', '▁Paris', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il cherche un itinéraire de Marseille à Toulouse', 'tokens': ['<s>', '▁Il', '▁cherche', '▁un', '▁itinéraire', '▁de', '▁Marseille', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut partir de Strasbourg pour un séjour à Rennes', 'tokens': ['<s>', '▁Elle', '▁veut', '▁partir', '▁de', '▁Strasbourg', '▁pour', '▁un', '▁séjour', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je prévois de prendre le bus de Nice à Paris', 'tokens': ['<s>', '▁Je', '▁pré', 'vois', '▁de', '▁prendre', '▁le', '▁bus', '▁de', '▁Nice', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de rejoindre ses amis de Lille à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁rejoindre', '▁ses', '▁amis', '▁de', '▁Lille', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous réservons un vol de Toulouse à Lyon', 'tokens': ['<s>', '▁Nous', '▁réserv', 'ons', '▁un', '▁vol', '▁de', '▁Toulouse', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il veut partir de Rennes pour visiter Marseille', 'tokens': ['<s>', '▁Il', '▁veut', '▁partir', '▁de', '▁Rennes', '▁pour', '▁visiter', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je prends un billet de train de Bordeaux à Strasbourg', 'tokens': ['<s>', '▁Je', '▁prends', '▁un', '▁billet', '▁de', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Lyon à Paris', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Lyon', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un billet pour un trajet de Lille à Nice', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁billet', '▁pour', '▁un', '▁trajet', '▁de', '▁Lille', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle doit rejoindre des collègues de Rennes à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁doit', '▁rejoindre', '▁des', '▁collègues', '▁de', '▁Rennes', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un vol de Marseille à Paris', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁vol', '▁de', '▁Marseille', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de partir de Lyon pour un événement à Toulouse', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁partir', '▁de', '▁Lyon', '▁pour', '▁un', '▁événement', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un train de Bordeaux à Lille', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Paris à Strasbourg', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Paris', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre des amis de Nice à Lyon', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Nice', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de voyager de Toulouse à Bordeaux', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Toulouse', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je pars de Rennes pour visiter Marseille', 'tokens': ['<s>', '▁Je', '▁par', 's', '▁de', '▁Rennes', '▁pour', '▁visiter', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit un trajet de Lyon à Paris', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Lyon', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Marseille à Nice', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Marseille', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il cherche un itinéraire de Bordeaux à Toulouse', 'tokens': ['<s>', '▁Il', '▁cherche', '▁un', '▁itinéraire', '▁de', '▁Bordeaux', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre ses amis de Lille à Rennes', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁ses', '▁amis', '▁de', '▁Lille', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de partir de Paris pour visiter Lyon', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁partir', '▁de', '▁Paris', '▁pour', '▁visiter', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un vol de Nice à Strasbourg', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁vol', '▁de', '▁Nice', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je dois aller de Marseille à Bordeaux', 'tokens': ['<s>', '▁Je', '▁dois', '▁aller', '▁de', '▁Marseille', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il veut un billet de train de Rennes à Toulouse', 'tokens': ['<s>', '▁Il', '▁veut', '▁un', '▁billet', '▁de', '▁train', '▁de', '▁Rennes', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Lyon à Marseille demain', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Lyon', '▁à', '▁Marseille', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Paris à Nice', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Paris', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit un trajet de Bordeaux à Rennes', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Bordeaux', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons demain de Toulouse à Lille', 'tokens': ['<s>', '▁Nous', '▁partons', '▁demain', '▁de', '▁Toulouse', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prend un billet pour un trajet de Paris à Strasbourg', 'tokens': ['<s>', '▁Il', '▁prend', '▁un', '▁billet', '▁pour', '▁un', '▁trajet', '▁de', '▁Paris', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit un voyage de Marseille à Lyon', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁un', '▁voyage', '▁de', '▁Marseille', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un itinéraire de train de Bordeaux à Paris', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁itinéraire', '▁de', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un billet pour un trajet de Lille à Marseille', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁billet', '▁pour', '▁un', '▁trajet', '▁de', '▁Lille', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Rennes à Toulouse', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Rennes', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de partir de Paris pour un événement à Nice', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁partir', '▁de', '▁Paris', '▁pour', '▁un', '▁événement', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un vol de Bordeaux à Lyon', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁vol', '▁de', '▁Bordeaux', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Marseille à Strasbourg', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Marseille', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de partir de Lille à Paris', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁partir', '▁de', '▁Lille', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un vol de Toulouse à Rennes', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁vol', '▁de', '▁Toulouse', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons en train de Bordeaux à Lyon', 'tokens': ['<s>', '▁Nous', '▁partons', '▁en', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut prendre un vol de Nice à Toulouse', 'tokens': ['<s>', '▁Elle', '▁veut', '▁prendre', '▁un', '▁vol', '▁de', '▁Nice', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de rejoindre des amis de Paris à Bordeaux', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Paris', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Lyon à Nice', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Lyon', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut partir de Marseille pour un séjour à Strasbourg', 'tokens': ['<s>', '▁Elle', '▁veut', '▁partir', '▁de', '▁Marseille', '▁pour', '▁un', '▁séjour', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je cherche un billet de train de Lille à Rennes', 'tokens': ['<s>', '▁Je', '▁cherche', '▁un', '▁billet', '▁de', '▁train', '▁de', '▁Lille', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de prendre le bus de Paris à Marseille', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁prendre', '▁le', '▁bus', '▁de', '▁Paris', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit un trajet de Bordeaux à Toulouse', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Bordeaux', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il part de Nice pour un événement à Lyon', 'tokens': ['<s>', '▁Il', '▁part', '▁de', '▁Nice', '▁pour', '▁un', '▁événement', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet pour un trajet de Rennes à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁pour', '▁un', '▁trajet', '▁de', '▁Rennes', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous réservons un vol de Paris à Strasbourg', 'tokens': ['<s>', '▁Nous', '▁réserv', 'ons', '▁un', '▁vol', '▁de', '▁Paris', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il cherche un itinéraire de Marseille à Nice', 'tokens': ['<s>', '▁Il', '▁cherche', '▁un', '▁itinéraire', '▁de', '▁Marseille', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut partir de Toulouse pour un séjour à Lille', 'tokens': ['<s>', '▁Elle', '▁veut', '▁partir', '▁de', '▁Toulouse', '▁pour', '▁un', '▁séjour', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de voyager de Rennes à Paris', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁voyager', '▁de', '▁Rennes', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un vol de Bordeaux à Marseille', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁vol', '▁de', '▁Bordeaux', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Je prévois de partir de Lyon pour visiter Toulouse', 'tokens': ['<s>', '▁Je', '▁pré', 'vois', '▁de', '▁partir', '▁de', '▁Lyon', '▁pour', '▁visiter', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre des collègues de Nice à Paris', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁des', '▁collègues', '▁de', '▁Nice', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Lille à Bordeaux', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Lille', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de voyager de Marseille à Lyon', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Marseille', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet de train de Rennes à Toulouse', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁de', '▁train', '▁de', '▁Rennes', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un trajet de Paris à Nice prévu', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁trajet', '▁de', '▁Paris', '▁à', '▁Nice', '▁prévu', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0], dtype=int64)}, {'text': 'Il prévoit un voyage de Marseille à Rennes', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁un', '▁voyage', '▁de', '▁Marseille', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet pour un trajet de Lille à Lyon', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁pour', '▁un', '▁trajet', '▁de', '▁Lille', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un vol de Bordeaux à Toulouse', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁vol', '▁de', '▁Bordeaux', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre des amis de Paris à Marseille', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Paris', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il cherche un itinéraire de Lyon à Strasbourg', 'tokens': ['<s>', '▁Il', '▁cherche', '▁un', '▁itinéraire', '▁de', '▁Lyon', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de prendre le train de Rennes à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁prendre', '▁le', '▁train', '▁de', '▁Rennes', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons en train de Toulouse à Nice', 'tokens': ['<s>', '▁Nous', '▁partons', '▁en', '▁train', '▁de', '▁Toulouse', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut prendre un billet de Marseille à Paris', 'tokens': ['<s>', '▁Elle', '▁veut', '▁prendre', '▁un', '▁billet', '▁de', '▁Marseille', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de partir de Lille pour visiter Lyon', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁partir', '▁de', '▁Lille', '▁pour', '▁visiter', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion de Bordeaux à Strasbourg', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁de', '▁Bordeaux', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Paris à Nice', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Paris', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle part de Rennes pour un séjour à Toulouse', 'tokens': ['<s>', '▁Elle', '▁part', '▁de', '▁Rennes', '▁pour', '▁un', '▁séjour', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Il cherche un vol de Marseille à Lyon', 'tokens': ['<s>', '▁Il', '▁cherche', '▁un', '▁vol', '▁de', '▁Marseille', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Bordeaux à Paris', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Bordeaux', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous réservons un vol de Lille à Marseille', 'tokens': ['<s>', '▁Nous', '▁réserv', 'ons', '▁un', '▁vol', '▁de', '▁Lille', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut partir de Toulouse pour un événement à Rennes', 'tokens': ['<s>', '▁Elle', '▁veut', '▁partir', '▁de', '▁Toulouse', '▁pour', '▁un', '▁événement', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons un trajet de Nice à Lyon', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁un', '▁trajet', '▁de', '▁Nice', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet pour un voyage de Paris à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁pour', '▁un', '▁voyage', '▁de', '▁Paris', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit de rejoindre des amis de Marseille à Nice', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁de', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Marseille', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Lille à Toulouse', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Lille', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un vol de Rennes à Lyon', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁vol', '▁de', '▁Rennes', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet pour un trajet de Bordeaux à Strasbourg', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁pour', '▁un', '▁trajet', '▁de', '▁Bordeaux', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Paris à Toulouse', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Paris', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de partir de Marseille pour visiter Rennes', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁partir', '▁de', '▁Marseille', '▁pour', '▁visiter', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Il prend un avion de Lyon à Lille', 'tokens': ['<s>', '▁Il', '▁prend', '▁un', '▁avion', '▁de', '▁Lyon', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Nice à Paris', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Nice', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons en train de Bordeaux à Marseille', 'tokens': ['<s>', '▁Nous', '▁partons', '▁en', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un vol pour un trajet de Toulouse à Strasbourg', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁vol', '▁pour', '▁un', '▁trajet', '▁de', '▁Toulouse', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Lille à Rennes', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Lille', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Marseille à Lyon', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Marseille', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il cherche un itinéraire de Paris à Bordeaux', 'tokens': ['<s>', '▁Il', '▁cherche', '▁un', '▁itinéraire', '▁de', '▁Paris', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet pour un trajet de Rennes à Nice', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁pour', '▁un', '▁trajet', '▁de', '▁Rennes', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un vol de Toulouse à Lille', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁vol', '▁de', '▁Toulouse', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre des amis de Lyon à Marseille', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Lyon', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Bordeaux à Paris', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Bordeaux', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle part de Nice pour un séjour à Lyon', 'tokens': ['<s>', '▁Elle', '▁part', '▁de', '▁Nice', '▁pour', '▁un', '▁séjour', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de partir de Rennes pour visiter Bordeaux', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁partir', '▁de', '▁Rennes', '▁pour', '▁visiter', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet de train pour un trajet de Toulouse à Paris', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁de', '▁train', '▁pour', '▁un', '▁trajet', '▁de', '▁Toulouse', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un vol de Lille à Lyon', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁vol', '▁de', '▁Lille', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Marseille à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Marseille', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit un trajet de Paris à Rennes', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Paris', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion pour un voyage de Nice à Toulouse', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁pour', '▁un', '▁voyage', '▁de', '▁Nice', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons en train de Bordeaux à Lyon', 'tokens': ['<s>', '▁Nous', '▁partons', '▁en', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Lille à Strasbourg', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Lille', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Paris à Marseille', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Paris', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre ses collègues de Rennes à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁ses', '▁collègues', '▁de', '▁Rennes', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Toulouse à Lyon', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Toulouse', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle partira de Nice pour un événement à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁partir', 'a', '▁de', '▁Nice', '▁pour', '▁un', '▁événement', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit un trajet de Lille à Paris', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Lille', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet pour un voyage de Marseille à Toulouse', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁pour', '▁un', '▁voyage', '▁de', '▁Marseille', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un vol de Rennes à Nice', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁vol', '▁de', '▁Rennes', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de rejoindre des amis de Paris à Lyon', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Paris', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de partir de Bordeaux pour visiter Marseille', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁partir', '▁de', '▁Bordeaux', '▁pour', '▁visiter', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet de train pour un trajet de Toulouse à Rennes', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁de', '▁train', '▁pour', '▁un', '▁trajet', '▁de', '▁Toulouse', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un vol de Lille à Marseille', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁vol', '▁de', '▁Lille', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Lyon à Paris', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Lyon', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit un trajet de Rennes à Strasbourg', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Rennes', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion pour un voyage de Bordeaux à Lille', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁pour', '▁un', '▁voyage', '▁de', '▁Bordeaux', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons en train de Nice à Toulouse', 'tokens': ['<s>', '▁Nous', '▁partons', '▁en', '▁train', '▁de', '▁Nice', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Paris à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Paris', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Marseille à Lyon', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Marseille', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre des collègues de Toulouse à Rennes', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁des', '▁collègues', '▁de', '▁Toulouse', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Lille à Nice', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Lille', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle partira de Lyon pour un événement à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁partir', 'a', '▁de', '▁Lyon', '▁pour', '▁un', '▁événement', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de partir de Rennes pour visiter Paris', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁partir', '▁de', '▁Rennes', '▁pour', '▁visiter', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet de train pour un trajet de Toulouse à Strasbourg', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁de', '▁train', '▁pour', '▁un', '▁trajet', '▁de', '▁Toulouse', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un vol de Bordeaux à Marseille', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁vol', '▁de', '▁Bordeaux', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Nice à Rennes', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Nice', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit un trajet de Paris à Lille', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Paris', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion pour un voyage de Marseille à Lyon', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁pour', '▁un', '▁voyage', '▁de', '▁Marseille', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons en train de Toulouse à Bordeaux', 'tokens': ['<s>', '▁Nous', '▁partons', '▁en', '▁train', '▁de', '▁Toulouse', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Rennes à Nice', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Rennes', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Lyon à Paris', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Lyon', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre des collègues de Bordeaux à Marseille', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁des', '▁collègues', '▁de', '▁Bordeaux', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Lille à Rennes', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Lille', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle partira de Nice pour un événement à Toulouse', 'tokens': ['<s>', '▁Elle', '▁partir', 'a', '▁de', '▁Nice', '▁pour', '▁un', '▁événement', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de partir de Paris pour visiter Lyon', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁partir', '▁de', '▁Paris', '▁pour', '▁visiter', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet de train pour un trajet de Bordeaux à Nice', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁de', '▁train', '▁pour', '▁un', '▁trajet', '▁de', '▁Bordeaux', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un vol de Marseille à Lille', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁vol', '▁de', '▁Marseille', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Toulouse à Strasbourg', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Toulouse', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit un trajet de Rennes à Paris', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Rennes', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion pour un voyage de Lyon à Marseille', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁pour', '▁un', '▁voyage', '▁de', '▁Lyon', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons en train de Bordeaux à Lille', 'tokens': ['<s>', '▁Nous', '▁partons', '▁en', '▁train', '▁de', '▁Bordeaux', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Nice à Toulouse', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Nice', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Paris à Rennes', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Paris', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre des amis de Marseille à Lyon', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Marseille', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Toulouse à Nice', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Toulouse', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle partira de Bordeaux pour un événement à Lyon', 'tokens': ['<s>', '▁Elle', '▁partir', 'a', '▁de', '▁Bordeaux', '▁pour', '▁un', '▁événement', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de partir de Lille pour visiter Paris', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁partir', '▁de', '▁Lille', '▁pour', '▁visiter', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet de train pour un trajet de Marseille à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁de', '▁train', '▁pour', '▁un', '▁trajet', '▁de', '▁Marseille', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un vol de Rennes à Strasbourg', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁vol', '▁de', '▁Rennes', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Nice à Lille', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Nice', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit un trajet de Paris à Marseille', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Paris', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion pour un voyage de Toulouse à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁pour', '▁un', '▁voyage', '▁de', '▁Toulouse', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons en train de Lyon à Nice', 'tokens': ['<s>', '▁Nous', '▁partons', '▁en', '▁train', '▁de', '▁Lyon', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Lille à Paris', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Lille', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Rennes à Marseille', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Rennes', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre des collègues de Bordeaux à Lyon', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁des', '▁collègues', '▁de', '▁Bordeaux', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Toulouse à Lille', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Toulouse', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle partira de Nice pour un événement à Paris', 'tokens': ['<s>', '▁Elle', '▁partir', 'a', '▁de', '▁Nice', '▁pour', '▁un', '▁événement', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de partir de Marseille pour visiter Rennes', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁partir', '▁de', '▁Marseille', '▁pour', '▁visiter', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet de train pour un trajet de Bordeaux à Lyon', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁de', '▁train', '▁pour', '▁un', '▁trajet', '▁de', '▁Bordeaux', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un vol de Lille à Nice', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁vol', '▁de', '▁Lille', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Toulouse à Bordeaux', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Toulouse', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit un trajet de Paris à Strasbourg', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Paris', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion pour un voyage de Rennes à Lyon', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁pour', '▁un', '▁voyage', '▁de', '▁Rennes', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons en train de Marseille à Lille', 'tokens': ['<s>', '▁Nous', '▁partons', '▁en', '▁train', '▁de', '▁Marseille', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Nice à Paris', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Nice', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Lyon à Rennes', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Lyon', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre des amis de Bordeaux à Marseille', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Bordeaux', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Toulouse à Paris', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Toulouse', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle partira de Nice pour un événement à Lyon', 'tokens': ['<s>', '▁Elle', '▁partir', 'a', '▁de', '▁Nice', '▁pour', '▁un', '▁événement', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de partir de Marseille pour visiter Lille', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁partir', '▁de', '▁Marseille', '▁pour', '▁visiter', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet de train pour un trajet de Bordeaux à Strasbourg', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁de', '▁train', '▁pour', '▁un', '▁trajet', '▁de', '▁Bordeaux', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un vol de Lille à Lyon', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁vol', '▁de', '▁Lille', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Toulouse à Rennes', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Toulouse', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit un trajet de Paris à Bordeaux', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Paris', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion pour un voyage de Nice à Lyon', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁pour', '▁un', '▁voyage', '▁de', '▁Nice', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons en train de Marseille à Paris', 'tokens': ['<s>', '▁Nous', '▁partons', '▁en', '▁train', '▁de', '▁Marseille', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Rennes à Strasbourg', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Rennes', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Lyon à Lille', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Lyon', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre des amis de Bordeaux à Paris', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Bordeaux', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Toulouse à Marseille', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Toulouse', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle partira de Nice pour un événement à Lille', 'tokens': ['<s>', '▁Elle', '▁partir', 'a', '▁de', '▁Nice', '▁pour', '▁un', '▁événement', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Nous prévoyons de partir de Paris pour visiter Bordeaux', 'tokens': ['<s>', '▁Nous', '▁pré', 'voy', 'ons', '▁de', '▁partir', '▁de', '▁Paris', '▁pour', '▁visiter', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un billet de train pour un trajet de Marseille à Rennes', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁billet', '▁de', '▁train', '▁pour', '▁un', '▁trajet', '▁de', '▁Marseille', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons un vol de Lille à Strasbourg', 'tokens': ['<s>', '▁Nous', '▁avons', '▁un', '▁vol', '▁de', '▁Lille', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut voyager de Toulouse à Lyon', 'tokens': ['<s>', '▁Elle', '▁veut', '▁voyager', '▁de', '▁Toulouse', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Il prévoit un trajet de Nice à Bordeaux', 'tokens': ['<s>', '▁Il', '▁prévoit', '▁un', '▁trajet', '▁de', '▁Nice', '▁à', '▁Bordeaux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prend un avion pour un voyage de Rennes à Toulouse', 'tokens': ['<s>', '▁Elle', '▁prend', '▁un', '▁avion', '▁pour', '▁un', '▁voyage', '▁de', '▁Rennes', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous partons en train de Paris à Lyon', 'tokens': ['<s>', '▁Nous', '▁partons', '▁en', '▁train', '▁de', '▁Paris', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle prévoit de voyager de Marseille à Lille', 'tokens': ['<s>', '▁Elle', '▁prévoit', '▁de', '▁voyager', '▁de', '▁Marseille', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous avons réservé un billet de Bordeaux à Paris', 'tokens': ['<s>', '▁Nous', '▁avons', '▁réservé', '▁un', '▁billet', '▁de', '▁Bordeaux', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle veut rejoindre des amis de Lyon à Nice', 'tokens': ['<s>', '▁Elle', '▁veut', '▁rejoindre', '▁des', '▁amis', '▁de', '▁Lyon', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Nous cherchons un vol de Lille à Marseille', 'tokens': ['<s>', '▁Nous', '▁cherchons', '▁un', '▁vol', '▁de', '▁Lille', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 2, 0], dtype=int64)}, {'text': 'Elle partira de Toulouse pour un événement à Paris', 'tokens': ['<s>', '▁Elle', '▁partir', 'a', '▁de', '▁Toulouse', '▁pour', '▁un', '▁événement', '▁à', '▁Paris', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe la journée à Bayonne, retour à Pau ensuite', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁journée', '▁à', '▁Bayonne', ',', '▁retour', '▁à', '▁Pau', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Orléans, retour à Blois demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Orléans', ',', '▁retour', '▁à', '▁Blois', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En visite à Chartres, retour prévu à Dreux ce soir', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Chartres', ',', '▁retour', '▁prévu', '▁à', '▁D', 'reux', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0, 0], dtype=int64)}, {'text': 'Je suis à Rennes pour une conférence, retour prévu à Saint-Malo demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Rennes', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁prévu', '▁à', '▁Saint', '-', 'Malo', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je suis en vacances à Saint-Nazaire, retour à Nantes ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Saint', '-', 'Nazaire', ',', '▁retour', '▁à', '▁Nantes', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En visite à Ajaccio, retour prévu à Bastia', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁', 'Ajaccio', ',', '▁retour', '▁prévu', '▁à', '▁Bastia', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement à Amiens pour le travail, retour prévu à Beauvais ensuite', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Amiens', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Beauvais', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Toulon, retour prévu à Marseille', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Toulon', ',', '▁retour', '▁prévu', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Thionville, retour prévu à Metz demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Thi', 'on', 'ville', ',', '▁retour', '▁prévu', '▁à', '▁Metz', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En vacances à Narbonne, retour prévu à Carcassonne', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Narbonne', ',', '▁retour', '▁prévu', '▁à', '▁Carcassonne', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Alençon pour le travail, retour prévu à Argentan ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Al', 'en', 'çon', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Argent', 'an', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En visite à Montauban, retour à Toulouse demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Montauban', ',', '▁retour', '▁à', '▁Toulouse', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Lannion, retour à Morlaix prévu demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁La', 'nni', 'on', ',', '▁retour', '▁à', '▁Morlaix', '▁prévu', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Actuellement en visite à Poitiers, je rentre à Niort ensuite', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Poitiers', ',', '▁je', '▁rentre', '▁à', '▁Niort', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Besançon, retour prévu à Dole', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Besançon', ',', '▁retour', '▁prévu', '▁à', '▁Dol', 'e', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis à Lorient pour quelques jours, retour prévu à Vannes', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Lorient', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Vannes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Mulhouse, retour prévu à Colmar ce soir', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Mulhouse', ',', '▁retour', '▁prévu', '▁à', '▁Colmar', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Lille, retour prévu à Roubaix', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Lille', ',', '▁retour', '▁prévu', '▁à', '▁Roubaix', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Arles, retour prévu à Nîmes', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁', 'Arles', ',', '▁retour', '▁prévu', '▁à', '▁Nîmes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en visite à Vichy, retour prévu à Moulins demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Vichy', ',', '▁retour', '▁prévu', '▁à', '▁Moulin', 's', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En déplacement à Périgueux, retour à Bergerac', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Périgueux', ',', '▁retour', '▁à', '▁Bergerac', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Tarbes pour le travail, retour à Pau prévu demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Tarbes', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁à', '▁Pau', '▁prévu', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'En visite à Toulon, retour prévu à Marseille ce soir', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Toulon', ',', '▁retour', '▁prévu', '▁à', '▁Marseille', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Actuellement à Paris pour une réunion, retour prévu à Versailles', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Paris', '▁pour', '▁une', '▁réunion', ',', '▁retour', '▁prévu', '▁à', '▁Versailles', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe la semaine à Rouen, retour prévu à Caen', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁semaine', '▁à', '▁Rouen', ',', '▁retour', '▁prévu', '▁à', '▁Caen', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Thonon-les-Bains, retour à Annecy', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Tho', 'non', '-', 'les', '-', 'Bains', ',', '▁retour', '▁à', '▁', 'Annecy', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis en vacances à Avignon, retour à Aix-en-Provence', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Avignon', ',', '▁retour', '▁à', '▁Aix', '-', 'en', '-', 'Provence', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 4, 4, 0], dtype=int64)}, {'text': 'En déplacement à Troyes, retour à Sens', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Troyes', ',', '▁retour', '▁à', '▁Sens', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Saint-Malo pour quelques jours, retour prévu à Rennes', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Saint', '-', 'Malo', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement à Reims pour le travail, retour à Châlons', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Reims', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁à', '▁Châ', 'lons', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En visite à Pau, retour prévu à Tarbes demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Pau', ',', '▁retour', '▁prévu', '▁à', '▁Tarbes', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en vacances à Saint-Quentin, retour prévu à Laon', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Saint', '-', 'Quentin', ',', '▁retour', '▁prévu', '▁à', '▁La', 'on', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Le Havre, retour à Rouen ensuite', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Le', '▁Havre', ',', '▁retour', '▁à', '▁Rouen', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Bayonne, retour prévu à Pau', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Bayonne', ',', '▁retour', '▁prévu', '▁à', '▁Pau', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Nancy, retour prévu à Metz', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Nancy', ',', '▁retour', '▁prévu', '▁à', '▁Metz', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Bourges pour le travail, retour prévu à Vierzon', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Bourges', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Vi', 'er', 'zon', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'En visite à Tours, retour prévu à Blois', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Tours', ',', '▁retour', '▁prévu', '▁à', '▁Blois', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement en visite à Ajaccio, retour prévu à Bastia', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁', 'Ajaccio', ',', '▁retour', '▁prévu', '▁à', '▁Bastia', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Quimper pour quelques jours, retour prévu à Brest', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Quimper', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Brest', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Toulouse, retour prévu à Albi', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Toulouse', ',', '▁retour', '▁prévu', '▁à', '▁', 'Albi', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En visite à Limoges, retour à Brive', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Limoges', ',', '▁retour', '▁à', '▁Brive', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Châteauroux, retour à Bourges', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Château', 'roux', ',', '▁retour', '▁à', '▁Bourges', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe la semaine à Orléans, retour prévu à Blois', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁semaine', '▁à', '▁Orléans', ',', '▁retour', '▁prévu', '▁à', '▁Blois', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Lorient pour une conférence, retour prévu à Vannes', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Lorient', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁prévu', '▁à', '▁Vannes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à La Rochelle, retour prévu à Rochefort', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁La', '▁Rochelle', ',', '▁retour', '▁prévu', '▁à', '▁Rochefort', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Calais, retour à Boulogne-sur-Mer', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Calais', ',', '▁retour', '▁à', '▁Boulogne', '-', 'sur', '-', 'Mer', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 4, 4, 0], dtype=int64)}, {'text': 'Actuellement en visite à Metz, retour à Nancy', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Metz', ',', '▁retour', '▁à', '▁Nancy', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Angoulême, retour prévu à Cognac', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁', 'Angoulême', ',', '▁retour', '▁prévu', '▁à', '▁Cognac', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Perpignan, retour à Narbonne', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Perpignan', ',', '▁retour', '▁à', '▁Narbonne', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe la journée à Brest, retour à Morlaix', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁journée', '▁à', '▁Brest', ',', '▁retour', '▁à', '▁Morlaix', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Dieppe, retour à Rouen', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Dieppe', ',', '▁retour', '▁à', '▁Rouen', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement en visite à Chambéry, retour à Grenoble', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Chambéry', ',', '▁retour', '▁à', '▁Grenoble', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Brive, retour à Tulle', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Brive', ',', '▁retour', '▁à', '▁Tu', 'lle', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Annemasse, retour à Thonon-les-Bains', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Anne', 'm', 'asse', ',', '▁retour', '▁à', '▁Tho', 'non', '-', 'les', '-', 'Bains', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 4, 4, 4, 4, 4, 0],
      dtype=int64)}, {'text': 'Actuellement en visite à Cannes, retour à Nice', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Cannes', ',', '▁retour', '▁à', '▁Nice', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe la semaine à Montluçon, retour prévu à Moulins', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁semaine', '▁à', '▁Mont', 'lu', 'çon', ',', '▁retour', '▁prévu', '▁à', '▁Moulin', 's', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis à Mulhouse pour le travail, retour prévu à Colmar', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Mulhouse', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Colmar', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Dijon, retour prévu à Beaune', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Dijon', ',', '▁retour', '▁prévu', '▁à', '▁Beau', 'ne', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis à Alès pour quelques jours, retour à Nîmes', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁A', 'lès', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁à', '▁Nîmes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En déplacement à Évreux, retour à Rouen', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁É', 'v', 'reux', ',', '▁retour', '▁à', '▁Rouen', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement en visite à Lens, retour à Arras', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Lens', ',', '▁retour', '▁à', '▁Ar', 'ras', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En vacances à Agen, retour prévu à Marmande', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁A', 'gen', ',', '▁retour', '▁prévu', '▁à', '▁Mar', 'mand', 'e', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'Je suis à Angers pour le travail, retour à Saumur', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Angers', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁à', '▁Saumur', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Morlaix, retour prévu à Brest', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Morlaix', ',', '▁retour', '▁prévu', '▁à', '▁Brest', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Albi, retour prévu à Castres', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁', 'Albi', ',', '▁retour', '▁prévu', '▁à', '▁Cast', 'res', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis en vacances à Cognac, retour à Angoulême', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Cognac', ',', '▁retour', '▁à', '▁', 'Angoulême', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Tulle, retour à Brive', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Tu', 'lle', ',', '▁retour', '▁à', '▁Brive', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Mende, retour à Rodez', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Men', 'de', ',', '▁retour', '▁à', '▁Rod', 'ez', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En vacances à Saintes, retour prévu à Rochefort', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Sainte', 's', ',', '▁retour', '▁prévu', '▁à', '▁Rochefort', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Pontoise pour une réunion, retour à Cergy', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Pont', 'oise', '▁pour', '▁une', '▁réunion', ',', '▁retour', '▁à', '▁C', 'er', 'gy', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Montargis, retour à Orléans', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Mont', 'arg', 'is', ',', '▁retour', '▁à', '▁Orléans', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Laon, retour prévu à Reims', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁La', 'on', ',', '▁retour', '▁prévu', '▁à', '▁Reims', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Bergerac, retour à Périgueux', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Bergerac', ',', '▁retour', '▁à', '▁Périgueux', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe la journée à Meaux, retour à Melun', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁journée', '▁à', '▁M', 'eaux', ',', '▁retour', '▁à', '▁Mel', 'un', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En déplacement à Guéret, retour à La Souterraine', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Gué', 'ret', ',', '▁retour', '▁à', '▁La', '▁Sou', 'terrain', 'e', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 4, 4, 0], dtype=int64)}, {'text': 'Actuellement en visite à Tarascon, retour à Avignon', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Tara', 's', 'con', ',', '▁retour', '▁à', '▁Avignon', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Draguignan, retour à Fréjus', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Dra', 'gu', 'ignan', ',', '▁retour', '▁à', '▁Fréjus', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Saint-Lô, retour à Coutances', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Saint', '-', 'L', 'ô', ',', '▁retour', '▁à', '▁Cout', 'ances', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Actuellement à Oyonnax, retour prévu à Bourg-en-Bresse', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁O', 'y', 'on', 'na', 'x', ',', '▁retour', '▁prévu', '▁à', '▁Bourg', '-', 'en', '-', 'B', 'resse', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 3, 3, 3, 0, 0, 0, 0, 2, 4, 4, 4, 4, 4, 0],
      dtype=int64)}, {'text': 'Je suis à Sète pour une conférence, retour prévu à Montpellier', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁S', 'ète', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁prévu', '▁à', '▁Montpellier', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Alès, retour à Nîmes', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁A', 'lès', ',', '▁retour', '▁à', '▁Nîmes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Châlons, retour à Reims', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Châ', 'lons', ',', '▁retour', '▁à', '▁Reims', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Montélimar, retour à Valence', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Mont', 'éli', 'mar', ',', '▁retour', '▁à', '▁Valence', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Figeac, retour à Cahors', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Fig', 'e', 'ac', ',', '▁retour', '▁à', '▁Ca', 'hors', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je passe la semaine à Maubeuge, retour à Valenciennes', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁semaine', '▁à', '▁M', 'aube', 'uge', ',', '▁retour', '▁à', '▁Valenciennes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Saint-Malo, retour à Rennes', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Saint', '-', 'Malo', ',', '▁retour', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement en visite à Aurillac, retour à Figeac', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Au', 'r', 'illac', ',', '▁retour', '▁à', '▁Fig', 'e', 'ac', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Lisieux, retour à Caen', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Li', 's', 'ieux', ',', '▁retour', '▁à', '▁Caen', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Abbeville, retour à Amiens', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Ab', 'be', 'ville', ',', '▁retour', '▁à', '▁Amiens', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Épinal pour quelques jours, retour à Nancy', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Ép', 'in', 'al', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁à', '▁Nancy', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement à Villeneuve-sur-Lot, retour à Agen', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Villeneuve', '-', 'sur', '-', 'L', 'ot', ',', '▁retour', '▁à', '▁A', 'gen', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Haguenau, retour à Strasbourg', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Ha', 'guen', 'au', ',', '▁retour', '▁à', '▁Strasbourg', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Auxerre, retour à Sens', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁', 'Auxerre', ',', '▁retour', '▁à', '▁Sens', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Issoire, retour à Clermont-Ferrand', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Is', 's', 'oire', ',', '▁retour', '▁à', '▁Clermont', '-', 'Ferrand', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Saint-Omer, retour à Dunkerque', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Saint', '-', 'O', 'mer', ',', '▁retour', '▁à', '▁Dunkerque', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Lannion, retour à Morlaix', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁La', 'nni', 'on', ',', '▁retour', '▁à', '▁Morlaix', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En déplacement pour le boulot à Amiens, mais je rentre à Beauvais ensuite', 'tokens': ['<s>', '▁En', '▁déplacement', '▁pour', '▁le', '▁boulot', '▁à', '▁Amiens', ',', '▁mais', '▁je', '▁rentre', '▁à', '▁Beauvais', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Actuellement chez un ami à Dijon, retour prévu à Mâcon demain', 'tokens': ['<s>', '▁Actuellement', '▁chez', '▁un', '▁ami', '▁à', '▁Dijon', ',', '▁retour', '▁prévu', '▁à', '▁M', 'â', 'con', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Marseille avant de retourner sur Arles', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Marseille', '▁avant', '▁de', '▁retourner', '▁sur', '▁', 'Arles', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis en visite à Lyon, mais je prévois de rentrer sur Villeurbanne', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Lyon', ',', '▁mais', '▁je', '▁pré', 'vois', '▁de', '▁rentrer', '▁sur', '▁Villeurbanne', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Pau, je rentre à Tarbes ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Pau', ',', '▁je', '▁rentre', '▁à', '▁Tarbes', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Actuellement en visite à Rennes, retour à Saint-Brieuc ce soir', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Rennes', ',', '▁retour', '▁à', '▁Saint', '-', 'Brieuc', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 0, 0, 0], dtype=int64)}, {'text': 'Je suis à Grenoble pour quelques jours, retour prévu à Chambéry ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Grenoble', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Chambéry', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En déplacement à Tours, je compte rentrer à Blois', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Tours', ',', '▁je', '▁compte', '▁rentrer', '▁à', '▁Blois', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Nice pour le travail, retour prévu à Menton demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Nice', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Men', 'ton', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Toulon, retour prévu à Marseille après-demain', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Toulon', ',', '▁retour', '▁prévu', '▁à', '▁Marseille', '▁après', '-', 'de', 'main', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je reste à Paris pour le moment, mais je rentre à Versailles bientôt', 'tokens': ['<s>', '▁Je', '▁reste', '▁à', '▁Paris', '▁pour', '▁le', '▁moment', ',', '▁mais', '▁je', '▁rentre', '▁à', '▁Versailles', '▁bientôt', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En visite à La Rochelle, retour à Rochefort ce week-end', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁La', '▁Rochelle', ',', '▁retour', '▁à', '▁Rochefort', '▁ce', '▁week', '-', 'end', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis à Nantes pour une réunion, retour à Angers ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Nantes', '▁pour', '▁une', '▁réunion', ',', '▁retour', '▁à', '▁Angers', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En vacances à Strasbourg, retour à Colmar demain', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Strasbourg', ',', '▁retour', '▁à', '▁Colmar', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en visite à Nîmes, retour prévu à Avignon ce soir', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Nîmes', ',', '▁retour', '▁prévu', '▁à', '▁Avignon', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'En déplacement à Poitiers, je rentre à Niort ensuite', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Poitiers', ',', '▁je', '▁rentre', '▁à', '▁Niort', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en vacances à Bordeaux, retour à Arcachon ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Bordeaux', ',', '▁retour', '▁à', '▁', 'Arcachon', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En visite à Lille, retour prévu à Dunkerque ce soir', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Lille', ',', '▁retour', '▁prévu', '▁à', '▁Dunkerque', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Actuellement à Limoges, je prévois de rentrer à Brive demain', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Limoges', ',', '▁je', '▁pré', 'vois', '▁de', '▁rentrer', '▁à', '▁Brive', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis à Rouen pour quelques jours, retour prévu à Caen ce soir', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Rouen', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Caen', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'En visite à Dijon, retour à Chalon-sur-Saône prévu demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Dijon', ',', '▁retour', '▁à', '▁Chalon', '-', 'sur', '-', 'Saône', '▁prévu', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 4, 4, 0, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Amiens, retour prévu à Beauvais ce soir', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Amiens', ',', '▁retour', '▁prévu', '▁à', '▁Beauvais', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Je passe la journée à Perpignan, retour prévu à Carcassonne', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁journée', '▁à', '▁Perpignan', ',', '▁retour', '▁prévu', '▁à', '▁Carcassonne', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Albi pour une conférence, retour à Castres ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁', 'Albi', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁à', '▁Cast', 'res', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En vacances à Cannes, retour prévu à Nice demain', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Cannes', ',', '▁retour', '▁prévu', '▁à', '▁Nice', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en visite à Lyon, retour prévu à Grenoble', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Lyon', ',', '▁retour', '▁prévu', '▁à', '▁Grenoble', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En déplacement à Saint-Étienne, je rentre à Lyon demain', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Saint', '-', 'Étienne', ',', '▁je', '▁rentre', '▁à', '▁Lyon', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en visite à Metz, retour prévu à Nancy ce soir', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Metz', ',', '▁retour', '▁prévu', '▁à', '▁Nancy', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Actuellement en visite à Mont-de-Marsan, je rentre à Dax ensuite', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Mont', '-', 'de', '-', 'Mar', 'san', ',', '▁je', '▁rentre', '▁à', '▁D', 'ax', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 0, 2, 4, 0, 0],
      dtype=int64)}, {'text': 'Je suis en déplacement à Lorient, retour prévu à Vannes ce soir', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Lorient', ',', '▁retour', '▁prévu', '▁à', '▁Vannes', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'En visite à Reims, retour prévu à Châlons demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Reims', ',', '▁retour', '▁prévu', '▁à', '▁Châ', 'lons', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Nevers, retour à Moulins ensuite', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Never', 's', ',', '▁retour', '▁à', '▁Moulin', 's', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je suis en vacances à Bastia, retour prévu à Ajaccio demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Bastia', ',', '▁retour', '▁prévu', '▁à', '▁', 'Ajaccio', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je reste à Pau pour l’instant, retour prévu à Tarbes ce soir', 'tokens': ['<s>', '▁Je', '▁reste', '▁à', '▁Pau', '▁pour', '▁l', '’', 'instant', ',', '▁retour', '▁prévu', '▁à', '▁Tarbes', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'En visite à Charleville-Mézières, retour prévu à Sedan demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Char', 'le', 'ville', '-', 'Mé', 'z', 'ières', ',', '▁retour', '▁prévu', '▁à', '▁Se', 'dan', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 2, 4, 0, 0],
      dtype=int64)}, {'text': 'Je suis à Clermont-Ferrand pour le travail, retour prévu à Moulins ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Clermont', '-', 'Ferrand', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Moulin', 's', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Actuellement en déplacement à Tours, retour prévu à Blois ce soir', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁déplacement', '▁à', '▁Tours', ',', '▁retour', '▁prévu', '▁à', '▁Blois', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Je suis en visite à Bourges, retour prévu à Vierzon demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Bourges', ',', '▁retour', '▁prévu', '▁à', '▁Vi', 'er', 'zon', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je passe la journée à Bayonne, retour à Pau ensuite', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁journée', '▁à', '▁Bayonne', ',', '▁retour', '▁à', '▁Pau', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Orléans, retour à Blois demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Orléans', ',', '▁retour', '▁à', '▁Blois', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En visite à Chartres, retour prévu à Dreux ce soir', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Chartres', ',', '▁retour', '▁prévu', '▁à', '▁D', 'reux', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0, 0], dtype=int64)}, {'text': 'Je suis à Rennes pour une conférence, retour prévu à Saint-Malo demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Rennes', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁prévu', '▁à', '▁Saint', '-', 'Malo', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je suis en vacances à Saint-Nazaire, retour à Nantes ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Saint', '-', 'Nazaire', ',', '▁retour', '▁à', '▁Nantes', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En visite à Ajaccio, retour prévu à Bastia', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁', 'Ajaccio', ',', '▁retour', '▁prévu', '▁à', '▁Bastia', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement à Amiens pour le travail, retour prévu à Beauvais ensuite', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Amiens', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Beauvais', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Toulon, retour prévu à Marseille', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Toulon', ',', '▁retour', '▁prévu', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Thionville, retour prévu à Metz demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Thi', 'on', 'ville', ',', '▁retour', '▁prévu', '▁à', '▁Metz', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En vacances à Narbonne, retour prévu à Carcassonne', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Narbonne', ',', '▁retour', '▁prévu', '▁à', '▁Carcassonne', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Alençon pour le travail, retour prévu à Argentan ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Al', 'en', 'çon', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Argent', 'an', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En visite à Montauban, retour à Toulouse demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Montauban', ',', '▁retour', '▁à', '▁Toulouse', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Lannion, retour à Morlaix prévu demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁La', 'nni', 'on', ',', '▁retour', '▁à', '▁Morlaix', '▁prévu', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Actuellement en visite à Poitiers, je rentre à Niort ensuite', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Poitiers', ',', '▁je', '▁rentre', '▁à', '▁Niort', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Besançon, retour prévu à Dole', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Besançon', ',', '▁retour', '▁prévu', '▁à', '▁Dol', 'e', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis à Lorient pour quelques jours, retour prévu à Vannes', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Lorient', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Vannes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Mulhouse, retour prévu à Colmar ce soir', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Mulhouse', ',', '▁retour', '▁prévu', '▁à', '▁Colmar', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Lille, retour prévu à Roubaix', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Lille', ',', '▁retour', '▁prévu', '▁à', '▁Roubaix', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Arles, retour prévu à Nîmes', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁', 'Arles', ',', '▁retour', '▁prévu', '▁à', '▁Nîmes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en visite à Vichy, retour prévu à Moulins demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Vichy', ',', '▁retour', '▁prévu', '▁à', '▁Moulin', 's', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En déplacement à Périgueux, retour à Bergerac', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Périgueux', ',', '▁retour', '▁à', '▁Bergerac', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Tarbes pour le travail, retour à Pau prévu demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Tarbes', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁à', '▁Pau', '▁prévu', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'En visite à Toulon, retour prévu à Marseille ce soir', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Toulon', ',', '▁retour', '▁prévu', '▁à', '▁Marseille', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Actuellement à Paris pour une réunion, retour prévu à Versailles', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Paris', '▁pour', '▁une', '▁réunion', ',', '▁retour', '▁prévu', '▁à', '▁Versailles', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe la semaine à Rouen, retour prévu à Caen', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁semaine', '▁à', '▁Rouen', ',', '▁retour', '▁prévu', '▁à', '▁Caen', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Thonon-les-Bains, retour à Annecy', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Tho', 'non', '-', 'les', '-', 'Bains', ',', '▁retour', '▁à', '▁', 'Annecy', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis en vacances à Avignon, retour à Aix-en-Provence', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Avignon', ',', '▁retour', '▁à', '▁Aix', '-', 'en', '-', 'Provence', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 4, 4, 0], dtype=int64)}, {'text': 'En déplacement à Troyes, retour à Sens', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Troyes', ',', '▁retour', '▁à', '▁Sens', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Saint-Malo pour quelques jours, retour prévu à Rennes', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Saint', '-', 'Malo', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement à Reims pour le travail, retour à Châlons', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Reims', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁à', '▁Châ', 'lons', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En visite à Pau, retour prévu à Tarbes demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Pau', ',', '▁retour', '▁prévu', '▁à', '▁Tarbes', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en vacances à Saint-Quentin, retour prévu à Laon', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Saint', '-', 'Quentin', ',', '▁retour', '▁prévu', '▁à', '▁La', 'on', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Le Havre, retour à Rouen ensuite', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Le', '▁Havre', ',', '▁retour', '▁à', '▁Rouen', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Bayonne, retour prévu à Pau', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Bayonne', ',', '▁retour', '▁prévu', '▁à', '▁Pau', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Nancy, retour prévu à Metz', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Nancy', ',', '▁retour', '▁prévu', '▁à', '▁Metz', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Bourges pour le travail, retour prévu à Vierzon', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Bourges', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Vi', 'er', 'zon', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'En visite à Tours, retour prévu à Blois', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Tours', ',', '▁retour', '▁prévu', '▁à', '▁Blois', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement en visite à Ajaccio, retour prévu à Bastia', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁', 'Ajaccio', ',', '▁retour', '▁prévu', '▁à', '▁Bastia', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Quimper pour quelques jours, retour prévu à Brest', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Quimper', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Brest', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Toulouse, retour prévu à Albi', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Toulouse', ',', '▁retour', '▁prévu', '▁à', '▁', 'Albi', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En visite à Limoges, retour à Brive', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Limoges', ',', '▁retour', '▁à', '▁Brive', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Châteauroux, retour à Bourges', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Château', 'roux', ',', '▁retour', '▁à', '▁Bourges', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe la semaine à Orléans, retour prévu à Blois', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁semaine', '▁à', '▁Orléans', ',', '▁retour', '▁prévu', '▁à', '▁Blois', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Lorient pour une conférence, retour prévu à Vannes', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Lorient', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁prévu', '▁à', '▁Vannes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à La Rochelle, retour prévu à Rochefort', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁La', '▁Rochelle', ',', '▁retour', '▁prévu', '▁à', '▁Rochefort', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Calais, retour à Boulogne-sur-Mer', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Calais', ',', '▁retour', '▁à', '▁Boulogne', '-', 'sur', '-', 'Mer', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 4, 4, 0], dtype=int64)}, {'text': 'Actuellement en visite à Metz, retour à Nancy', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Metz', ',', '▁retour', '▁à', '▁Nancy', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Angoulême, retour prévu à Cognac', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁', 'Angoulême', ',', '▁retour', '▁prévu', '▁à', '▁Cognac', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Perpignan, retour à Narbonne', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Perpignan', ',', '▁retour', '▁à', '▁Narbonne', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Saint-Omer, retour à Dunkerque', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Saint', '-', 'O', 'mer', ',', '▁retour', '▁à', '▁Dunkerque', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Lannion, retour à Morlaix', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁La', 'nni', 'on', ',', '▁retour', '▁à', '▁Morlaix', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En déplacement pour le boulot à Amiens, mais je rentre à Beauvais ensuite', 'tokens': ['<s>', '▁En', '▁déplacement', '▁pour', '▁le', '▁boulot', '▁à', '▁Amiens', ',', '▁mais', '▁je', '▁rentre', '▁à', '▁Beauvais', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Actuellement chez un ami à Dijon, retour prévu à Mâcon demain', 'tokens': ['<s>', '▁Actuellement', '▁chez', '▁un', '▁ami', '▁à', '▁Dijon', ',', '▁retour', '▁prévu', '▁à', '▁M', 'â', 'con', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Marseille avant de retourner sur Arles', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Marseille', '▁avant', '▁de', '▁retourner', '▁sur', '▁', 'Arles', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis en visite à Lyon, mais je prévois de rentrer sur Villeurbanne', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Lyon', ',', '▁mais', '▁je', '▁pré', 'vois', '▁de', '▁rentrer', '▁sur', '▁Villeurbanne', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Pau, je rentre à Tarbes ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Pau', ',', '▁je', '▁rentre', '▁à', '▁Tarbes', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Actuellement en visite à Rennes, retour à Saint-Brieuc ce soir', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Rennes', ',', '▁retour', '▁à', '▁Saint', '-', 'Brieuc', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 0, 0, 0], dtype=int64)}, {'text': 'Je suis à Grenoble pour quelques jours, retour prévu à Chambéry ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Grenoble', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Chambéry', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En déplacement à Tours, je compte rentrer à Blois', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Tours', ',', '▁je', '▁compte', '▁rentrer', '▁à', '▁Blois', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Nice pour le travail, retour prévu à Menton demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Nice', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Men', 'ton', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Toulon, retour prévu à Marseille après-demain', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Toulon', ',', '▁retour', '▁prévu', '▁à', '▁Marseille', '▁après', '-', 'de', 'main', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je reste à Paris pour le moment, mais je rentre à Versailles bientôt', 'tokens': ['<s>', '▁Je', '▁reste', '▁à', '▁Paris', '▁pour', '▁le', '▁moment', ',', '▁mais', '▁je', '▁rentre', '▁à', '▁Versailles', '▁bientôt', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En visite à La Rochelle, retour à Rochefort ce week-end', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁La', '▁Rochelle', ',', '▁retour', '▁à', '▁Rochefort', '▁ce', '▁week', '-', 'end', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis à Nantes pour une réunion, retour à Angers ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Nantes', '▁pour', '▁une', '▁réunion', ',', '▁retour', '▁à', '▁Angers', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En vacances à Strasbourg, retour à Colmar demain', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Strasbourg', ',', '▁retour', '▁à', '▁Colmar', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en visite à Nîmes, retour prévu à Avignon ce soir', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Nîmes', ',', '▁retour', '▁prévu', '▁à', '▁Avignon', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'En déplacement à Poitiers, je rentre à Niort ensuite', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Poitiers', ',', '▁je', '▁rentre', '▁à', '▁Niort', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en vacances à Bordeaux, retour à Arcachon ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Bordeaux', ',', '▁retour', '▁à', '▁', 'Arcachon', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En visite à Lille, retour prévu à Dunkerque ce soir', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Lille', ',', '▁retour', '▁prévu', '▁à', '▁Dunkerque', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Actuellement à Limoges, je prévois de rentrer à Brive demain', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Limoges', ',', '▁je', '▁pré', 'vois', '▁de', '▁rentrer', '▁à', '▁Brive', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis à Rouen pour quelques jours, retour prévu à Caen ce soir', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Rouen', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Caen', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'En visite à Dijon, retour à Chalon-sur-Saône prévu demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Dijon', ',', '▁retour', '▁à', '▁Chalon', '-', 'sur', '-', 'Saône', '▁prévu', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 4, 4, 0, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Amiens, retour prévu à Beauvais ce soir', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Amiens', ',', '▁retour', '▁prévu', '▁à', '▁Beauvais', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Je passe la journée à Perpignan, retour prévu à Carcassonne', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁journée', '▁à', '▁Perpignan', ',', '▁retour', '▁prévu', '▁à', '▁Carcassonne', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Albi pour une conférence, retour à Castres ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁', 'Albi', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁à', '▁Cast', 'res', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En vacances à Cannes, retour prévu à Nice demain', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Cannes', ',', '▁retour', '▁prévu', '▁à', '▁Nice', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en visite à Lyon, retour prévu à Grenoble', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Lyon', ',', '▁retour', '▁prévu', '▁à', '▁Grenoble', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En déplacement à Saint-Étienne, je rentre à Lyon demain', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Saint', '-', 'Étienne', ',', '▁je', '▁rentre', '▁à', '▁Lyon', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en visite à Metz, retour prévu à Nancy ce soir', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Metz', ',', '▁retour', '▁prévu', '▁à', '▁Nancy', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Actuellement en visite à Mont-de-Marsan, je rentre à Dax ensuite', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Mont', '-', 'de', '-', 'Mar', 'san', ',', '▁je', '▁rentre', '▁à', '▁D', 'ax', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 0, 2, 4, 0, 0],
      dtype=int64)}, {'text': 'Je suis en déplacement à Lorient, retour prévu à Vannes ce soir', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Lorient', ',', '▁retour', '▁prévu', '▁à', '▁Vannes', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'En visite à Reims, retour prévu à Châlons demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Reims', ',', '▁retour', '▁prévu', '▁à', '▁Châ', 'lons', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Nevers, retour à Moulins ensuite', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Never', 's', ',', '▁retour', '▁à', '▁Moulin', 's', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je suis en vacances à Bastia, retour prévu à Ajaccio demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Bastia', ',', '▁retour', '▁prévu', '▁à', '▁', 'Ajaccio', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je reste à Pau pour l’instant, retour prévu à Tarbes ce soir', 'tokens': ['<s>', '▁Je', '▁reste', '▁à', '▁Pau', '▁pour', '▁l', '’', 'instant', ',', '▁retour', '▁prévu', '▁à', '▁Tarbes', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'En visite à Charleville-Mézières, retour prévu à Sedan demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Char', 'le', 'ville', '-', 'Mé', 'z', 'ières', ',', '▁retour', '▁prévu', '▁à', '▁Se', 'dan', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 2, 4, 0, 0],
      dtype=int64)}, {'text': 'Je suis à Clermont-Ferrand pour le travail, retour prévu à Moulins ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Clermont', '-', 'Ferrand', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Moulin', 's', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Actuellement en déplacement à Tours, retour prévu à Blois ce soir', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁déplacement', '▁à', '▁Tours', ',', '▁retour', '▁prévu', '▁à', '▁Blois', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Je suis en visite à Bourges, retour prévu à Vierzon demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Bourges', ',', '▁retour', '▁prévu', '▁à', '▁Vi', 'er', 'zon', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je passe la journée à Bayonne, retour à Pau ensuite', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁journée', '▁à', '▁Bayonne', ',', '▁retour', '▁à', '▁Pau', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Orléans, retour à Blois demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Orléans', ',', '▁retour', '▁à', '▁Blois', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En visite à Chartres, retour prévu à Dreux ce soir', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Chartres', ',', '▁retour', '▁prévu', '▁à', '▁D', 'reux', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0, 0], dtype=int64)}, {'text': 'Je suis à Rennes pour une conférence, retour prévu à Saint-Malo demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Rennes', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁prévu', '▁à', '▁Saint', '-', 'Malo', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je suis en vacances à Saint-Nazaire, retour à Nantes ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Saint', '-', 'Nazaire', ',', '▁retour', '▁à', '▁Nantes', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En visite à Ajaccio, retour prévu à Bastia', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁', 'Ajaccio', ',', '▁retour', '▁prévu', '▁à', '▁Bastia', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement à Amiens pour le travail, retour prévu à Beauvais ensuite', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Amiens', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Beauvais', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Toulon, retour prévu à Marseille', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Toulon', ',', '▁retour', '▁prévu', '▁à', '▁Marseille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Thionville, retour prévu à Metz demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Thi', 'on', 'ville', ',', '▁retour', '▁prévu', '▁à', '▁Metz', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En vacances à Narbonne, retour prévu à Carcassonne', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Narbonne', ',', '▁retour', '▁prévu', '▁à', '▁Carcassonne', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Alençon pour le travail, retour prévu à Argentan ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Al', 'en', 'çon', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Argent', 'an', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En visite à Montauban, retour à Toulouse demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Montauban', ',', '▁retour', '▁à', '▁Toulouse', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Lannion, retour à Morlaix prévu demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁La', 'nni', 'on', ',', '▁retour', '▁à', '▁Morlaix', '▁prévu', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Actuellement en visite à Poitiers, je rentre à Niort ensuite', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Poitiers', ',', '▁je', '▁rentre', '▁à', '▁Niort', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Besançon, retour prévu à Dole', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Besançon', ',', '▁retour', '▁prévu', '▁à', '▁Dol', 'e', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis à Lorient pour quelques jours, retour prévu à Vannes', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Lorient', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Vannes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Mulhouse, retour prévu à Colmar ce soir', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Mulhouse', ',', '▁retour', '▁prévu', '▁à', '▁Colmar', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Lille, retour prévu à Roubaix', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Lille', ',', '▁retour', '▁prévu', '▁à', '▁Roubaix', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Arles, retour prévu à Nîmes', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁', 'Arles', ',', '▁retour', '▁prévu', '▁à', '▁Nîmes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en visite à Vichy, retour prévu à Moulins demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Vichy', ',', '▁retour', '▁prévu', '▁à', '▁Moulin', 's', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En déplacement à Périgueux, retour à Bergerac', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Périgueux', ',', '▁retour', '▁à', '▁Bergerac', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Tarbes pour le travail, retour à Pau prévu demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Tarbes', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁à', '▁Pau', '▁prévu', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'En visite à Toulon, retour prévu à Marseille ce soir', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Toulon', ',', '▁retour', '▁prévu', '▁à', '▁Marseille', '▁ce', '▁soir', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Actuellement à Paris pour une réunion, retour prévu à Versailles', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Paris', '▁pour', '▁une', '▁réunion', ',', '▁retour', '▁prévu', '▁à', '▁Versailles', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe la semaine à Rouen, retour prévu à Caen', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁semaine', '▁à', '▁Rouen', ',', '▁retour', '▁prévu', '▁à', '▁Caen', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Thonon-les-Bains, retour à Annecy', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Tho', 'non', '-', 'les', '-', 'Bains', ',', '▁retour', '▁à', '▁', 'Annecy', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis en vacances à Avignon, retour à Aix-en-Provence', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Avignon', ',', '▁retour', '▁à', '▁Aix', '-', 'en', '-', 'Provence', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 4, 4, 0], dtype=int64)}, {'text': 'En déplacement à Troyes, retour à Sens', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Troyes', ',', '▁retour', '▁à', '▁Sens', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Saint-Malo pour quelques jours, retour prévu à Rennes', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Saint', '-', 'Malo', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement à Reims pour le travail, retour à Châlons', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Reims', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁à', '▁Châ', 'lons', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En visite à Pau, retour prévu à Tarbes demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Pau', ',', '▁retour', '▁prévu', '▁à', '▁Tarbes', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en vacances à Saint-Quentin, retour prévu à Laon', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁Saint', '-', 'Quentin', ',', '▁retour', '▁prévu', '▁à', '▁La', 'on', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Le Havre, retour à Rouen ensuite', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Le', '▁Havre', ',', '▁retour', '▁à', '▁Rouen', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Bayonne, retour prévu à Pau', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Bayonne', ',', '▁retour', '▁prévu', '▁à', '▁Pau', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Nancy, retour prévu à Metz', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Nancy', ',', '▁retour', '▁prévu', '▁à', '▁Metz', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Bourges pour le travail, retour prévu à Vierzon', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Bourges', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Vi', 'er', 'zon', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'En visite à Tours, retour prévu à Blois', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Tours', ',', '▁retour', '▁prévu', '▁à', '▁Blois', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement en visite à Ajaccio, retour prévu à Bastia', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁', 'Ajaccio', ',', '▁retour', '▁prévu', '▁à', '▁Bastia', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Quimper pour quelques jours, retour prévu à Brest', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Quimper', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁prévu', '▁à', '▁Brest', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Toulouse, retour prévu à Albi', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Toulouse', ',', '▁retour', '▁prévu', '▁à', '▁', 'Albi', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En visite à Limoges, retour à Brive', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Limoges', ',', '▁retour', '▁à', '▁Brive', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Châteauroux, retour à Bourges', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Château', 'roux', ',', '▁retour', '▁à', '▁Bourges', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe la semaine à Orléans, retour prévu à Blois', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁semaine', '▁à', '▁Orléans', ',', '▁retour', '▁prévu', '▁à', '▁Blois', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Lorient pour une conférence, retour prévu à Vannes', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Lorient', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁prévu', '▁à', '▁Vannes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à La Rochelle, retour prévu à Rochefort', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁La', '▁Rochelle', ',', '▁retour', '▁prévu', '▁à', '▁Rochefort', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Calais, retour à Boulogne-sur-Mer', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Calais', ',', '▁retour', '▁à', '▁Boulogne', '-', 'sur', '-', 'Mer', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 4, 4, 0], dtype=int64)}, {'text': 'Actuellement en visite à Metz, retour à Nancy', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Metz', ',', '▁retour', '▁à', '▁Nancy', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Angoulême, retour prévu à Cognac', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁', 'Angoulême', ',', '▁retour', '▁prévu', '▁à', '▁Cognac', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Perpignan, retour à Narbonne', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Perpignan', ',', '▁retour', '▁à', '▁Narbonne', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Saint-Omer, retour à Dunkerque', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Saint', '-', 'O', 'mer', ',', '▁retour', '▁à', '▁Dunkerque', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en vacances à Lannion, retour à Morlaix', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁La', 'nni', 'on', ',', '▁retour', '▁à', '▁Morlaix', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En déplacement pour le boulot à Amiens, mais je rentre à Beauvais ensuite', 'tokens': ['<s>', '▁En', '▁déplacement', '▁pour', '▁le', '▁boulot', '▁à', '▁Amiens', ',', '▁mais', '▁je', '▁rentre', '▁à', '▁Beauvais', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Actuellement chez un ami à Dijon, retour prévu à Mâcon demain', 'tokens': ['<s>', '▁Actuellement', '▁chez', '▁un', '▁ami', '▁à', '▁Dijon', ',', '▁retour', '▁prévu', '▁à', '▁M', 'â', 'con', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Marseille avant de retourner sur Arles', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Marseille', '▁avant', '▁de', '▁retourner', '▁sur', '▁', 'Arles', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis en visite à Lyon, mais je prévois de rentrer sur Villeurbanne', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Lyon', ',', '▁mais', '▁je', '▁pré', 'vois', '▁de', '▁rentrer', '▁sur', '▁Villeurbanne', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Pau, je rentre à Tarbes ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Pau', ',', '▁je', '▁rentre', '▁à', '▁Tarbes', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Brest, retour à Lorient demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Brest', ',', '▁retour', '▁à', '▁Lorient', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Nice, retour à Cannes ensuite', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Nice', ',', '▁retour', '▁à', '▁Cannes', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En vacances à Menton, retour à Nice prévu demain', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Men', 'ton', ',', '▁retour', '▁à', '▁Nice', '▁prévu', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0, 0, 0], dtype=int64)}, {'text': 'Je suis à Limoges pour le travail, retour prévu à Angoulême', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Limoges', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁', 'Angoulême', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En visite à Calais, retour prévu à Dunkerque', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Calais', ',', '▁retour', '▁prévu', '▁à', '▁Dunkerque', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Reims pour une conférence, retour à Épernay ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Reims', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁à', '▁Ép', 'er', 'nay', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'En déplacement à Thonon-les-Bains, retour prévu à Chambéry', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Tho', 'non', '-', 'les', '-', 'Bains', ',', '▁retour', '▁prévu', '▁à', '▁Chambéry', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Cahors, retour à Figeac ensuite', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Ca', 'hors', ',', '▁retour', '▁à', '▁Fig', 'e', 'ac', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je suis en visite à Valence, retour prévu à Montélimar', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Valence', ',', '▁retour', '▁prévu', '▁à', '▁Mont', 'éli', 'mar', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'En vacances à Fréjus, retour à Toulon ensuite', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Fréjus', ',', '▁retour', '▁à', '▁Toulon', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Actuellement à Ajaccio, retour prévu à Calvi', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁', 'Ajaccio', ',', '▁retour', '▁prévu', '▁à', '▁Cal', 'vi', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Rodez, retour prévu à Aurillac', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Rod', 'ez', ',', '▁retour', '▁prévu', '▁à', '▁Au', 'r', 'illac', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'En visite à Digne-les-Bains, retour prévu à Manosque', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁D', 'igne', '-', 'les', '-', 'Bains', ',', '▁retour', '▁prévu', '▁à', '▁Man', 'osque', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis à Saint-Brieuc pour une réunion, retour à Lorient demain', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Saint', '-', 'Brieuc', '▁pour', '▁une', '▁réunion', ',', '▁retour', '▁à', '▁Lorient', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En déplacement pour le travail à Béziers, retour prévu à Narbonne', 'tokens': ['<s>', '▁En', '▁déplacement', '▁pour', '▁le', '▁travail', '▁à', '▁Béziers', ',', '▁retour', '▁prévu', '▁à', '▁Narbonne', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe la semaine à Vesoul, retour prévu à Luxeuil-les-Bains', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁semaine', '▁à', '▁Ve', 's', 'oul', ',', '▁retour', '▁prévu', '▁à', '▁Lux', 'euil', '-', 'les', '-', 'Bains', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 4, 4, 4, 4, 4, 0],
      dtype=int64)}, {'text': 'Je suis en vacances à Évreux, retour prévu à Vernon', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁É', 'v', 'reux', ',', '▁retour', '▁prévu', '▁à', '▁Vern', 'on', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En visite à Troyes, retour à Chaumont ensuite', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Troyes', ',', '▁retour', '▁à', '▁Chau', 'mont', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je suis à Bourg-en-Bresse pour le travail, retour à Lyon ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Bourg', '-', 'en', '-', 'B', 'resse', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁à', '▁Lyon', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 2, 0, 0],
      dtype=int64)}, {'text': 'En déplacement à Dreux, retour prévu à Chartres', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁D', 'reux', ',', '▁retour', '▁prévu', '▁à', '▁Chartres', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Saint-Étienne pour une conférence, retour à Clermont-Ferrand', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Saint', '-', 'Étienne', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁à', '▁Clermont', '-', 'Ferrand', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'En vacances à Nîmes, retour prévu à Montpellier', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Nîmes', ',', '▁retour', '▁prévu', '▁à', '▁Montpellier', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Chambéry, retour à Annecy ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Chambéry', ',', '▁retour', '▁à', '▁', 'Annecy', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En visite à Périgueux, retour à Bergerac demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Périgueux', ',', '▁retour', '▁à', '▁Bergerac', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Actuellement à Vannes, retour prévu à Lorient', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Vannes', ',', '▁retour', '▁prévu', '▁à', '▁Lorient', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Charleville-Mézières, retour prévu à Sedan', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Char', 'le', 'ville', '-', 'Mé', 'z', 'ières', ',', '▁retour', '▁prévu', '▁à', '▁Se', 'dan', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 2, 4, 0],
      dtype=int64)}, {'text': 'Je suis à Perpignan pour une conférence, retour à Narbonne ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Perpignan', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁à', '▁Narbonne', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'En déplacement à Pau, retour prévu à Bayonne', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Pau', ',', '▁retour', '▁prévu', '▁à', '▁Bayonne', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis à Roanne pour le travail, retour prévu à Lyon', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Ro', 'anne', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Lyon', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Saint-Nazaire, retour à La Baule', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Saint', '-', 'Nazaire', ',', '▁retour', '▁à', '▁La', '▁Bau', 'le', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'En visite à Morlaix, retour prévu à Saint-Brieuc', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Morlaix', ',', '▁retour', '▁prévu', '▁à', '▁Saint', '-', 'Brieuc', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Vichy, retour prévu à Clermont-Ferrand', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Vichy', ',', '▁retour', '▁prévu', '▁à', '▁Clermont', '-', 'Ferrand', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'Actuellement en visite à Orléans, retour à Blois ensuite', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Orléans', ',', '▁retour', '▁à', '▁Blois', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Besançon, retour à Dole', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Besançon', ',', '▁retour', '▁à', '▁Dol', 'e', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En vacances à Ajaccio, retour prévu à Porto-Vecchio', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁', 'Ajaccio', ',', '▁retour', '▁prévu', '▁à', '▁Porto', '-', 'V', 'e', 'cchi', 'o', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 4, 4, 4, 4, 0], dtype=int64)}, {'text': 'Je passe la journée à Douai, retour prévu à Lille', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁journée', '▁à', '▁Dou', 'ai', ',', '▁retour', '▁prévu', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en visite à Foix, retour à Pamiers ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁visite', '▁à', '▁Foi', 'x', ',', '▁retour', '▁à', '▁Pam', 'iers', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En déplacement à Vesoul, retour prévu à Besançon', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Ve', 's', 'oul', ',', '▁retour', '▁prévu', '▁à', '▁Besançon', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement à Caen pour une réunion, retour prévu à Bayeux', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Caen', '▁pour', '▁une', '▁réunion', ',', '▁retour', '▁prévu', '▁à', '▁Bay', 'eux', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En vacances à Brest, retour prévu à Quimper', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Brest', ',', '▁retour', '▁prévu', '▁à', '▁Quimper', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Saint-Omer, retour à Boulogne-sur-Mer', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Saint', '-', 'O', 'mer', ',', '▁retour', '▁à', '▁Boulogne', '-', 'sur', '-', 'Mer', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 3, 0, 0, 0, 2, 4, 4, 4, 4, 0],
      dtype=int64)}, {'text': 'En visite à Agen, retour prévu à Marmande demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁A', 'gen', ',', '▁retour', '▁prévu', '▁à', '▁Mar', 'mand', 'e', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je suis à Épinal pour le travail, retour à Nancy', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Ép', 'in', 'al', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁à', '▁Nancy', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement en déplacement à Saint-Dizier, retour à Vitry-le-François', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁déplacement', '▁à', '▁Saint', '-', 'Di', 'zier', ',', '▁retour', '▁à', '▁Vitry', '-', 'le', '-', 'François', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 3, 3, 0, 0, 0, 2, 4, 4, 4, 4, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Laval, retour à Mayenne', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Laval', ',', '▁retour', '▁à', '▁Mayenne', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Bayonne, retour prévu à Biarritz', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Bayonne', ',', '▁retour', '▁prévu', '▁à', '▁Biarritz', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Thonon-les-Bains, retour à Annecy', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Tho', 'non', '-', 'les', '-', 'Bains', ',', '▁retour', '▁à', '▁', 'Annecy', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En visite à Alençon, retour prévu à Le Mans demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Al', 'en', 'çon', ',', '▁retour', '▁prévu', '▁à', '▁Le', '▁Mans', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je suis à Rochefort pour une conférence, retour à La Rochelle ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Rochefort', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁à', '▁La', '▁Rochelle', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Actuellement à Arras, retour prévu à Douai', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Ar', 'ras', ',', '▁retour', '▁prévu', '▁à', '▁Dou', 'ai', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je passe la journée à Albi, retour à Castres', 'tokens': ['<s>', '▁Je', '▁passe', '▁la', '▁journée', '▁à', '▁', 'Albi', ',', '▁retour', '▁à', '▁Cast', 'res', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En déplacement à Saint-Malo, retour prévu à Rennes', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Saint', '-', 'Malo', ',', '▁retour', '▁prévu', '▁à', '▁Rennes', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Brive, retour prévu à Tulle', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Brive', ',', '▁retour', '▁prévu', '▁à', '▁Tu', 'lle', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis à Guéret pour le travail, retour prévu à Limoges', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Gué', 'ret', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁à', '▁Limoges', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Châteauroux, retour à Bourges demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Château', 'roux', ',', '▁retour', '▁à', '▁Bourges', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Actuellement en déplacement à Laval, retour à Angers', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁déplacement', '▁à', '▁Laval', ',', '▁retour', '▁à', '▁Angers', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Bourges, retour prévu à Vierzon', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Bourges', ',', '▁retour', '▁prévu', '▁à', '▁Vi', 'er', 'zon', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Périgueux, retour à Bergerac', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Périgueux', ',', '▁retour', '▁à', '▁Bergerac', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Cahors, retour prévu à Montauban', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Ca', 'hors', ',', '▁retour', '▁prévu', '▁à', '▁Montauban', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Tulle, retour prévu à Brive', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Tu', 'lle', ',', '▁retour', '▁prévu', '▁à', '▁Brive', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement en visite à Carcassonne, retour à Narbonne', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁Carcassonne', ',', '▁retour', '▁à', '▁Narbonne', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Saint-Gaudens, retour prévu à Toulouse', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Saint', '-', 'G', 'aud', 'ens', ',', '▁retour', '▁prévu', '▁à', '▁Toulouse', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Issoire, retour à Clermont-Ferrand', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Is', 's', 'oire', ',', '▁retour', '▁à', '▁Clermont', '-', 'Ferrand', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'En visite à Montélimar, retour à Valence', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Mont', 'éli', 'mar', ',', '▁retour', '▁à', '▁Valence', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement à Saint-Affrique, retour prévu à Millau', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁Saint', '-', 'A', 'ff', 'rique', ',', '▁retour', '▁prévu', '▁à', '▁Mill', 'au', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 3, 3, 3, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Mende, retour à Florac', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Men', 'de', ',', '▁retour', '▁à', '▁Flora', 'c', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En vacances à Villeneuve-sur-Lot, retour à Agen', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Villeneuve', '-', 'sur', '-', 'L', 'ot', ',', '▁retour', '▁à', '▁A', 'gen', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis à Castres pour une conférence, retour à Albi ensuite', 'tokens': ['<s>', '▁Je', '▁suis', '▁à', '▁Cast', 'res', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁à', '▁', 'Albi', '▁ensuite', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'En déplacement à Langres, retour à Chaumont', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Lang', 'res', ',', '▁retour', '▁à', '▁Chau', 'mont', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'En visite à Oyonnax, retour prévu à Nantua', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁O', 'y', 'on', 'na', 'x', ',', '▁retour', '▁prévu', '▁à', '▁Nan', 'tu', 'a', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Digne-les-Bains, retour à Sisteron', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁D', 'igne', '-', 'les', '-', 'Bains', ',', '▁retour', '▁à', '▁Si', 'ster', 'on', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 2, 4, 4, 0],
      dtype=int64)}, {'text': 'Actuellement en déplacement à Montauban, retour à Cahors', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁déplacement', '▁à', '▁Montauban', ',', '▁retour', '▁à', '▁Ca', 'hors', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis en vacances à Angoulême, retour prévu à Cognac', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁vacances', '▁à', '▁', 'Angoulême', ',', '▁retour', '▁prévu', '▁à', '▁Cognac', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Saumur, retour à Tours', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Saumur', ',', '▁retour', '▁à', '▁Tours', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En déplacement à Thionville, retour à Metz', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Thi', 'on', 'ville', ',', '▁retour', '▁à', '▁Metz', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'Actuellement à Albi, retour prévu à Carmaux', 'tokens': ['<s>', '▁Actuellement', '▁à', '▁', 'Albi', ',', '▁retour', '▁prévu', '▁à', '▁Car', 'm', 'aux', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Forbach, retour à Metz', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁For', 'bach', ',', '▁retour', '▁à', '▁Metz', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Vannes, retour prévu à Auray', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Vannes', ',', '▁retour', '▁prévu', '▁à', '▁Au', 'ray', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Pontivy, retour à Vannes', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Pont', 'iv', 'y', ',', '▁retour', '▁à', '▁Vannes', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Sarlat, retour à Bergerac demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁Sar', 'lat', ',', '▁retour', '▁à', '▁Bergerac', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Actuellement en déplacement à Le Creusot, retour à Chalon-sur-Saône', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁déplacement', '▁à', '▁Le', '▁Cre', 'us', 'ot', ',', '▁retour', '▁à', '▁Chalon', '-', 'sur', '-', 'Saône', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 3, 3, 0, 0, 0, 2, 4, 4, 4, 4, 0], dtype=int64)}, {'text': 'En vacances à Paray-le-Monial, retour à Mâcon', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Para', 'y', '-', 'le', '-', 'Mon', 'ial', ',', '▁retour', '▁à', '▁M', 'â', 'con', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 3, 0, 0, 0, 2, 4, 4, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Maubeuge, retour à Lille', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁M', 'aube', 'uge', ',', '▁retour', '▁à', '▁Lille', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à Soissons, retour à Laon demain', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁So', 'issons', ',', '▁retour', '▁à', '▁La', 'on', '▁demain', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Montluçon, retour à Moulins', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Mont', 'lu', 'çon', ',', '▁retour', '▁à', '▁Moulin', 's', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Actuellement en visite à Dax, retour à Bayonne', 'tokens': ['<s>', '▁Actuellement', '▁en', '▁visite', '▁à', '▁D', 'ax', ',', '▁retour', '▁à', '▁Bayonne', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En vacances à Avranches, retour prévu à Saint-Lô', 'tokens': ['<s>', '▁En', '▁vacances', '▁à', '▁Avr', 'anche', 's', ',', '▁retour', '▁prévu', '▁à', '▁Saint', '-', 'L', 'ô', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 4, 4, 4, 0], dtype=int64)}, {'text': 'Je suis en déplacement à Vierzon, retour à Bourges', 'tokens': ['<s>', '▁Je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Vi', 'er', 'zon', ',', '▁retour', '▁à', '▁Bourges', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En visite à La Flèche, retour prévu à Le Mans', 'tokens': ['<s>', '▁En', '▁visite', '▁à', '▁La', '▁Fl', 'èche', ',', '▁retour', '▁prévu', '▁à', '▁Le', '▁Mans', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je passe quelques jours à Vendôme, retour prévu à Blois', 'tokens': ['<s>', '▁Je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Vend', 'ôme', ',', '▁retour', '▁prévu', '▁à', '▁Blois', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0], dtype=int64)}, {'text': 'En déplacement à Villers-Cotterêts, retour à Soissons', 'tokens': ['<s>', '▁En', '▁déplacement', '▁à', '▁Villers', '-', 'Co', 'tter', 'êt', 's', ',', '▁retour', '▁à', '▁So', 'issons', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 2, 4, 0], dtype=int64)}, {'text': 'Je viens de Lyon, je suis actuellement à Marseille et je veux rentrer chez moi.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Lyon', ',', '▁je', '▁suis', '▁actuellement', '▁à', '▁Marseille', '▁et', '▁je', '▁veux', '▁rentrer', '▁chez', '▁moi', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis originaire de Bordeaux, maintenant en déplacement à Lille, je souhaite rentrer à la maison.', 'tokens': ['<s>', '▁Je', '▁suis', '▁originaire', '▁de', '▁Bordeaux', ',', '▁maintenant', '▁en', '▁déplacement', '▁à', '▁Lille', ',', '▁je', '▁souhaite', '▁rentrer', '▁à', '▁la', '▁maison', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis né à Strasbourg, mais actuellement en vacances à Nice, retour prévu bientôt.', 'tokens': ['<s>', '▁Je', '▁suis', '▁né', '▁à', '▁Strasbourg', ',', '▁mais', '▁actuellement', '▁en', '▁vacances', '▁à', '▁Nice', ',', '▁retour', '▁prévu', '▁bientôt', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Toulouse, actuellement à Paris pour un séminaire, retour à la maison demain.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Toulouse', ',', '▁actuellement', '▁à', '▁Paris', '▁pour', '▁un', '▁séminaire', ',', '▁retour', '▁à', '▁la', '▁maison', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Nantes, mais je passe le weekend à Rennes, retour à mon domicile après-demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Nantes', ',', '▁mais', '▁je', '▁passe', '▁le', '▁weekend', '▁à', '▁Rennes', ',', '▁retour', '▁à', '▁mon', '▁domicile', '▁après', '-', 'de', 'main', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0], dtype=int64)}, {'text': 'Je viens d’Avignon et je suis à Montpellier pour le travail, je rentre chez moi demain soir.', 'tokens': ['<s>', '▁Je', '▁viens', '▁d', '’', 'Avignon', '▁et', '▁je', '▁suis', '▁à', '▁Montpellier', '▁pour', '▁le', '▁travail', ',', '▁je', '▁rentre', '▁chez', '▁moi', '▁demain', '▁soir', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0], dtype=int64)}, {'text': 'Je suis originaire de Grenoble, actuellement en visite à Lyon, retour prévu chez moi.', 'tokens': ['<s>', '▁Je', '▁suis', '▁originaire', '▁de', '▁Grenoble', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Lyon', ',', '▁retour', '▁prévu', '▁chez', '▁moi', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Metz, en déplacement à Reims, je rentre dès que possible.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Metz', ',', '▁en', '▁déplacement', '▁à', '▁Reims', ',', '▁je', '▁rentre', '▁dès', '▁que', '▁possible', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je vis à Dijon, mais je suis actuellement à Besançon et je retourne bientôt chez moi.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Dijon', ',', '▁mais', '▁je', '▁suis', '▁actuellement', '▁à', '▁Besançon', '▁et', '▁je', '▁retourne', '▁bientôt', '▁chez', '▁moi', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis né à Angers et en visite à Tours, je prévois de rentrer chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁né', '▁à', '▁Angers', '▁et', '▁en', '▁visite', '▁à', '▁Tours', ',', '▁je', '▁pré', 'vois', '▁de', '▁rentrer', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je viens de Perpignan, actuellement à Toulouse pour un concert, retour chez moi ensuite.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Perpignan', ',', '▁actuellement', '▁à', '▁Toulouse', '▁pour', '▁un', '▁concert', ',', '▁retour', '▁chez', '▁moi', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Toulon, en visite à Marseille, je retourne à la maison demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Toulon', ',', '▁en', '▁visite', '▁à', '▁Marseille', ',', '▁je', '▁retourne', '▁à', '▁la', '▁maison', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis originaire de Nancy, actuellement à Metz pour une conférence, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁originaire', '▁de', '▁Nancy', ',', '▁actuellement', '▁à', '▁Metz', '▁pour', '▁une', '▁conférence', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à La Rochelle, actuellement à Bordeaux pour quelques jours, je rentre ensuite.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁La', '▁Rochelle', ',', '▁actuellement', '▁à', '▁Bordeaux', '▁pour', '▁quelques', '▁jours', ',', '▁je', '▁rentre', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Clermont-Ferrand, en déplacement à Lyon, retour prévu chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Clermont', '-', 'Ferrand', ',', '▁en', '▁déplacement', '▁à', '▁Lyon', ',', '▁retour', '▁prévu', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je viens de Limoges, actuellement à Poitiers pour un séminaire, je retourne à la maison.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Limoges', ',', '▁actuellement', '▁à', '▁Poitiers', '▁pour', '▁un', '▁séminaire', ',', '▁je', '▁retourne', '▁à', '▁la', '▁maison', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Bayonne, en vacances à Biarritz, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Bayonne', ',', '▁en', '▁vacances', '▁à', '▁Biarritz', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Pau, actuellement à Tarbes pour une réunion, retour chez moi ensuite.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Pau', ',', '▁actuellement', '▁à', '▁Tarbes', '▁pour', '▁une', '▁réunion', ',', '▁retour', '▁chez', '▁moi', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Brest, mais je suis en déplacement à Rennes, retour à domicile demain.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Brest', ',', '▁mais', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Rennes', ',', '▁retour', '▁à', '▁domicile', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je viens de La Roche-sur-Yon, actuellement en visite à Nantes, je retourne chez moi ensuite.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁La', '▁Roche', '-', 'sur', '-', 'Y', 'on', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Nantes', ',', '▁je', '▁retourne', '▁chez', '▁moi', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0,
       0, 0, 0], dtype=int64)}, {'text': 'Je vis à Lille, mais je passe quelques jours à Dunkerque, retour à domicile bientôt.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Lille', ',', '▁mais', '▁je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Dunkerque', ',', '▁retour', '▁à', '▁domicile', '▁bientôt', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Nîmes, en déplacement à Montpellier pour une réunion, je rentre après.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Nîmes', ',', '▁en', '▁déplacement', '▁à', '▁Montpellier', '▁pour', '▁une', '▁réunion', ',', '▁je', '▁rentre', '▁après', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis originaire de Saint-Nazaire, actuellement en vacances à La Baule, retour chez moi ensuite.', 'tokens': ['<s>', '▁Je', '▁suis', '▁originaire', '▁de', '▁Saint', '-', 'Nazaire', ',', '▁actuellement', '▁en', '▁vacances', '▁à', '▁La', '▁Bau', 'le', ',', '▁retour', '▁chez', '▁moi', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 2, 4, 4, 0, 0, 0, 0, 0, 0,
       0], dtype=int64)}, {'text': 'Je suis de Chartres, en visite à Dreux, retour chez moi après-demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Chartres', ',', '▁en', '▁visite', '▁à', '▁D', 'reux', ',', '▁retour', '▁chez', '▁moi', '▁après', '-', 'de', 'main', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Perpignan, mais actuellement à Carcassonne, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Perpignan', ',', '▁mais', '▁actuellement', '▁à', '▁Carcassonne', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je vis à Mulhouse, en déplacement à Colmar, retour prévu chez moi ensuite.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Mulhouse', ',', '▁en', '▁déplacement', '▁à', '▁Colmar', ',', '▁retour', '▁prévu', '▁chez', '▁moi', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Poitiers, actuellement à Châtellerault pour un rendez-vous, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Poitiers', ',', '▁actuellement', '▁à', '▁Châtel', 'le', 'rault', '▁pour', '▁un', '▁rendez', '-', 'vous', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0], dtype=int64)}, {'text': 'Je suis de Saint-Malo, actuellement en visite à Dinard, retour chez moi prévu demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Saint', '-', 'Malo', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Din', 'ard', ',', '▁retour', '▁chez', '▁moi', '▁prévu', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Versailles, en déplacement à Paris, retour prévu chez moi bientôt.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Versailles', ',', '▁en', '▁déplacement', '▁à', '▁Paris', ',', '▁retour', '▁prévu', '▁chez', '▁moi', '▁bientôt', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je viens de Rodez, actuellement en déplacement à Millau, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Rod', 'ez', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Mill', 'au', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Cannes, en visite à Nice pour quelques jours, retour chez moi prévu.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Cannes', ',', '▁en', '▁visite', '▁à', '▁Nice', '▁pour', '▁quelques', '▁jours', ',', '▁retour', '▁chez', '▁moi', '▁prévu', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Thonon-les-Bains, actuellement en visite à Annecy, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Tho', 'non', '-', 'les', '-', 'Bains', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁', 'Annecy', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0,
       0, 0], dtype=int64)}, {'text': 'Je vis à Mende, en déplacement à Florac pour le travail, retour prévu chez moi.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Men', 'de', ',', '▁en', '▁déplacement', '▁à', '▁Flora', 'c', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁prévu', '▁chez', '▁moi', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis originaire de Belfort, actuellement à Montbéliard, retour chez moi ensuite.', 'tokens': ['<s>', '▁Je', '▁suis', '▁originaire', '▁de', '▁Belfort', ',', '▁actuellement', '▁à', '▁Mont', 'b', 'éli', 'ard', ',', '▁retour', '▁chez', '▁moi', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je viens de Sedan, en déplacement à Charleville-Mézières, retour chez moi prévu demain.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Se', 'dan', ',', '▁en', '▁déplacement', '▁à', '▁Char', 'le', 'ville', '-', 'Mé', 'z', 'ières', ',', '▁retour', '▁chez', '▁moi', '▁prévu', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0,
       0, 0, 0], dtype=int64)}, {'text': 'Je suis de Cognac, actuellement en visite à Angoulême, retour prévu chez moi ensuite.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Cognac', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁', 'Angoulême', ',', '▁retour', '▁prévu', '▁chez', '▁moi', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Vienne, actuellement en déplacement à Lyon, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Vienne', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Lyon', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Saumur, en déplacement à Tours pour le travail, retour chez moi ensuite.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Saumur', ',', '▁en', '▁déplacement', '▁à', '▁Tours', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁chez', '▁moi', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Creil, en déplacement à Compiègne, retour prévu chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Cre', 'il', ',', '▁en', '▁déplacement', '▁à', '▁Compiègne', ',', '▁retour', '▁prévu', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Ajaccio, actuellement en visite à Bastia, retour prévu chez moi demain.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁', 'Ajaccio', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Bastia', ',', '▁retour', '▁prévu', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je viens de Saint-Étienne, actuellement en déplacement à Lyon, retour chez moi prévu demain.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Saint', '-', 'Étienne', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Lyon', ',', '▁retour', '▁chez', '▁moi', '▁prévu', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Lorient, actuellement en visite à Vannes, retour prévu chez moi ensuite.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Lorient', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Vannes', ',', '▁retour', '▁prévu', '▁chez', '▁moi', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Laval, actuellement en déplacement à Mayenne, retour chez moi prévu.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Laval', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Mayenne', ',', '▁retour', '▁chez', '▁moi', '▁prévu', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Vesoul, actuellement en visite à Belfort, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Ve', 's', 'oul', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Belfort', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je viens de Montluçon, actuellement en déplacement à Moulins, retour chez moi ensuite.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Mont', 'lu', 'çon', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Moulin', 's', ',', '▁retour', '▁chez', '▁moi', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Figeac, actuellement en déplacement à Cahors, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Fig', 'e', 'ac', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Ca', 'hors', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Verdun, actuellement en visite à Bar-le-Duc, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Verdun', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Bar', '-', 'le', '-', 'D', 'uc', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0,
       0], dtype=int64)}, {'text': 'Je suis originaire de La Flèche, actuellement en déplacement à Angers, retour chez moi.', 'tokens': ['<s>', '▁Je', '▁suis', '▁originaire', '▁de', '▁La', '▁Fl', 'èche', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Angers', ',', '▁retour', '▁chez', '▁moi', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Charleville-Mézières, en déplacement à Sedan, retour chez moi ensuite.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Char', 'le', 'ville', '-', 'Mé', 'z', 'ières', ',', '▁en', '▁déplacement', '▁à', '▁Se', 'dan', ',', '▁retour', '▁chez', '▁moi', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0,
       0, 0], dtype=int64)}, {'text': 'Je suis de Saintes, actuellement en visite à La Rochelle, retour chez moi prévu.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Sainte', 's', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁La', '▁Rochelle', ',', '▁retour', '▁chez', '▁moi', '▁prévu', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Quimper, actuellement en déplacement à Brest, retour chez moi ensuite.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Quimper', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Brest', ',', '▁retour', '▁chez', '▁moi', '▁ensuite', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je viens de Bourg-en-Bresse, actuellement à Lyon, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Bourg', '-', 'en', '-', 'B', 'resse', ',', '▁actuellement', '▁à', '▁Lyon', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Alès, en déplacement à Nîmes pour le travail, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁A', 'lès', ',', '▁en', '▁déplacement', '▁à', '▁Nîmes', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis originaire de Tarbes, actuellement en visite à Pau, retour chez moi.', 'tokens': ['<s>', '▁Je', '▁suis', '▁originaire', '▁de', '▁Tarbes', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Pau', ',', '▁retour', '▁chez', '▁moi', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Dax, en déplacement à Mont-de-Marsan, retour chez moi prévu.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁D', 'ax', ',', '▁en', '▁déplacement', '▁à', '▁Mont', '-', 'de', '-', 'Mar', 'san', ',', '▁retour', '▁chez', '▁moi', '▁prévu', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0,
       0], dtype=int64)}, {'text': 'Je viens de Vannes, actuellement en visite à Lorient, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Vannes', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Lorient', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je vis à Périgueux, actuellement en déplacement à Bergerac, retour chez moi.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Périgueux', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Bergerac', ',', '▁retour', '▁chez', '▁moi', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Mâcon, en déplacement à Chalon-sur-Saône, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁M', 'â', 'con', ',', '▁en', '▁déplacement', '▁à', '▁Chalon', '-', 'sur', '-', 'Saône', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0,
       0], dtype=int64)}, {'text': 'Je suis de Meaux, actuellement à Melun pour une réunion, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁M', 'eaux', ',', '▁actuellement', '▁à', '▁Mel', 'un', '▁pour', '▁une', '▁réunion', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Laon, en déplacement à Soissons, retour chez moi prévu.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁La', 'on', ',', '▁en', '▁déplacement', '▁à', '▁So', 'issons', ',', '▁retour', '▁chez', '▁moi', '▁prévu', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de La Roche-sur-Yon, actuellement en déplacement à Cholet, retour chez moi.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁La', '▁Roche', '-', 'sur', '-', 'Y', 'on', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Cho', 'let', ',', '▁retour', '▁chez', '▁moi', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0,
       0, 0], dtype=int64)}, {'text': 'Je viens de Cherbourg, actuellement à Saint-Lô pour le travail, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Cherbourg', ',', '▁actuellement', '▁à', '▁Saint', '-', 'L', 'ô', '▁pour', '▁le', '▁travail', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Les Sables-d’Olonne, en déplacement à La Roche-sur-Yon, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Les', '▁S', 'ables', '-', 'd', '’', 'O', 'l', 'onne', ',', '▁en', '▁déplacement', '▁à', '▁La', '▁Roche', '-', 'sur', '-', 'Y', 'on', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 2, 4, 4, 4, 4,
       4, 4, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Châteauroux, actuellement en visite à Issoudun, retour chez moi.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Château', 'roux', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Is', 's', 'oud', 'un', ',', '▁retour', '▁chez', '▁moi', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 2, 4, 4, 4, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis originaire de Mont-de-Marsan, actuellement en déplacement à Dax, retour chez moi prévu demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁originaire', '▁de', '▁Mont', '-', 'de', '-', 'Mar', 'san', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁D', 'ax', ',', '▁retour', '▁chez', '▁moi', '▁prévu', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0,
       0, 0, 0, 0], dtype=int64)}, {'text': 'Je vis à Argentan, actuellement en déplacement à Alençon, retour chez moi.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Argent', 'an', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Al', 'en', 'çon', ',', '▁retour', '▁chez', '▁moi', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 2, 4, 4, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Cambrai, en visite à Douai, retour chez moi prévu.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Cambrai', ',', '▁en', '▁visite', '▁à', '▁Dou', 'ai', ',', '▁retour', '▁chez', '▁moi', '▁prévu', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je viens de Sens, actuellement à Auxerre pour un rendez-vous, retour chez moi.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Sens', ',', '▁actuellement', '▁à', '▁', 'Auxerre', '▁pour', '▁un', '▁rendez', '-', 'vous', ',', '▁retour', '▁chez', '▁moi', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Gaillac, en déplacement à Albi, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Ga', 'illac', ',', '▁en', '▁déplacement', '▁à', '▁', 'Albi', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Beauvais, en déplacement à Amiens, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Beauvais', ',', '▁en', '▁déplacement', '▁à', '▁Amiens', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je viens de Pontarlier, actuellement à Besançon pour une formation, retour chez moi demain.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Pont', 'ar', 'lier', ',', '▁actuellement', '▁à', '▁Besançon', '▁pour', '▁une', '▁formation', ',', '▁retour', '▁chez', '▁moi', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis originaire de Fécamp, en déplacement à Rouen, retour chez moi prévu demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁originaire', '▁de', '▁Fé', 'camp', ',', '▁en', '▁déplacement', '▁à', '▁Rouen', ',', '▁retour', '▁chez', '▁moi', '▁prévu', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Villefranche-sur-Saône, actuellement en visite à Lyon, retour chez moi.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Villefranche', '-', 'sur', '-', 'Saône', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Lyon', ',', '▁retour', '▁chez', '▁moi', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je viens de Saint-Omer, actuellement en déplacement à Calais, retour chez moi prévu demain.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Saint', '-', 'O', 'mer', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Calais', ',', '▁retour', '▁chez', '▁moi', '▁prévu', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Originaire de Marseille, je suis à Paris pour le travail et je rentre demain.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Marseille', ',', '▁je', '▁suis', '▁à', '▁Paris', '▁pour', '▁le', '▁travail', '▁et', '▁je', '▁rentre', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': "J'habite à Lyon et je suis actuellement à Strasbourg pour quelques jours.", 'tokens': ['<s>', '▁J', "'", 'habite', '▁à', '▁Lyon', '▁et', '▁je', '▁suis', '▁actuellement', '▁à', '▁Strasbourg', '▁pour', '▁quelques', '▁jours', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Bordeaux, en voyage à Nantes, retour prévu après-demain.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Bordeaux', ',', '▁en', '▁voyage', '▁à', '▁Nantes', ',', '▁retour', '▁prévu', '▁après', '-', 'de', 'main', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Lille et je suis à Dunkerque pour le weekend.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Lille', '▁et', '▁je', '▁suis', '▁à', '▁Dunkerque', '▁pour', '▁le', '▁weekend', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je viens de Toulouse et je passe quelques jours à Marseille.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Toulouse', '▁et', '▁je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Marseille', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Originaire de Rouen, je suis à Caen pour un rendez-vous, retour prévu demain.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Rouen', ',', '▁je', '▁suis', '▁à', '▁Caen', '▁pour', '▁un', '▁rendez', '-', 'vous', ',', '▁retour', '▁prévu', '▁demain', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Nîmes et je visite Avignon pour la journée.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Nîmes', '▁et', '▁je', '▁visite', '▁Avignon', '▁pour', '▁la', '▁journée', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': "J'habite à Tours, mais je suis à Poitiers pour le travail cette semaine.", 'tokens': ['<s>', '▁J', "'", 'habite', '▁à', '▁Tours', ',', '▁mais', '▁je', '▁suis', '▁à', '▁Poitiers', '▁pour', '▁le', '▁travail', '▁cette', '▁semaine', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Besançon, actuellement en déplacement à Dijon.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Besançon', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Dijon', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je vis à Brest, mais je suis à Lorient pour une conférence.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Brest', ',', '▁mais', '▁je', '▁suis', '▁à', '▁Lorient', '▁pour', '▁une', '▁conférence', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Originaire de Metz, je passe quelques jours à Nancy.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Metz', ',', '▁je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Nancy', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis de Mulhouse, en visite à Strasbourg pour le weekend.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Mulhouse', ',', '▁en', '▁visite', '▁à', '▁Strasbourg', '▁pour', '▁le', '▁weekend', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je viens de Tarbes et je suis actuellement à Pau pour un rendez-vous.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Tarbes', '▁et', '▁je', '▁suis', '▁actuellement', '▁à', '▁Pau', '▁pour', '▁un', '▁rendez', '-', 'vous', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0], dtype=int64)}, {'text': "J'habite à La Rochelle, mais je suis en déplacement à Rochefort.", 'tokens': ['<s>', '▁J', "'", 'habite', '▁à', '▁La', '▁Rochelle', ',', '▁mais', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Rochefort', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis originaire de Dunkerque, en déplacement à Lille pour le travail.', 'tokens': ['<s>', '▁Je', '▁suis', '▁originaire', '▁de', '▁Dunkerque', ',', '▁en', '▁déplacement', '▁à', '▁Lille', '▁pour', '▁le', '▁travail', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je vis à Chalon-sur-Saône et je suis actuellement à Dijon pour un séminaire.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Chalon', '-', 'sur', '-', 'Saône', '▁et', '▁je', '▁suis', '▁actuellement', '▁à', '▁Dijon', '▁pour', '▁un', '▁séminaire', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': "Originaire d'Annecy, je suis à Chambéry pour une réunion.", 'tokens': ['<s>', '▁Originaire', '▁d', "'", 'Annecy', ',', '▁je', '▁suis', '▁à', '▁Chambéry', '▁pour', '▁une', '▁réunion', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je viens de Clermont-Ferrand, en déplacement à Saint-Étienne pour deux jours.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Clermont', '-', 'Ferrand', ',', '▁en', '▁déplacement', '▁à', '▁Saint', '-', 'Étienne', '▁pour', '▁deux', '▁jours', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 4, 4, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Chartres, mais je suis en déplacement à Dreux pour le travail.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Chartres', ',', '▁mais', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁D', 'reux', '▁pour', '▁le', '▁travail', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Narbonne et je suis en visite à Perpignan pour un événement.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Narbonne', '▁et', '▁je', '▁suis', '▁en', '▁visite', '▁à', '▁Perpignan', '▁pour', '▁un', '▁événement', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Originaire de Lorient, je passe quelques jours à Quimper.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Lorient', ',', '▁je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Quimper', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je viens de Mende et je suis en déplacement à Millau.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Men', 'de', '▁et', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Mill', 'au', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je vis à Bayonne, mais je suis actuellement à Dax pour un rendez-vous.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Bayonne', ',', '▁mais', '▁je', '▁suis', '▁actuellement', '▁à', '▁D', 'ax', '▁pour', '▁un', '▁rendez', '-', 'vous', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': "J'habite à Alençon et je suis en déplacement à Argentan.", 'tokens': ['<s>', '▁J', "'", 'habite', '▁à', '▁Al', 'en', 'çon', '▁et', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Argent', 'an', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Originaire de Rodez, je suis actuellement à Cahors pour un projet.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Rod', 'ez', ',', '▁je', '▁suis', '▁actuellement', '▁à', '▁Ca', 'hors', '▁pour', '▁un', '▁projet', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Vannes, mais je passe la semaine à Lorient.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Vannes', ',', '▁mais', '▁je', '▁passe', '▁la', '▁semaine', '▁à', '▁Lorient', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je viens de Périgueux, actuellement à Bergerac pour un séminaire.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Périgueux', ',', '▁actuellement', '▁à', '▁Bergerac', '▁pour', '▁un', '▁séminaire', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Pau, en visite à Bayonne pour quelques jours.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Pau', ',', '▁en', '▁visite', '▁à', '▁Bayonne', '▁pour', '▁quelques', '▁jours', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': "J'habite à Saint-Brieuc, mais je suis en déplacement à Brest.", 'tokens': ['<s>', '▁J', "'", 'habite', '▁à', '▁Saint', '-', 'Brieuc', ',', '▁mais', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Brest', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je vis à Avignon, mais je passe quelques jours à Arles.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Avignon', ',', '▁mais', '▁je', '▁passe', '▁quelques', '▁jours', '▁à', '▁', 'Arles', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Originaire de Biarritz, actuellement en déplacement à Bayonne.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Biarritz', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Bayonne', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je viens de Sens, en déplacement à Auxerre.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Sens', ',', '▁en', '▁déplacement', '▁à', '▁', 'Auxerre', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je suis de Vesoul et je suis en visite à Belfort.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Ve', 's', 'oul', '▁et', '▁je', '▁suis', '▁en', '▁visite', '▁à', '▁Belfort', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je vis à Thionville, mais je suis en déplacement à Metz.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Thi', 'on', 'ville', ',', '▁mais', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Metz', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Originaire de Montauban, je suis à Cahors pour une réunion.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Montauban', ',', '▁je', '▁suis', '▁à', '▁Ca', 'hors', '▁pour', '▁une', '▁réunion', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Forbach et je suis actuellement à Sarreguemines.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁For', 'bach', '▁et', '▁je', '▁suis', '▁actuellement', '▁à', '▁S', 'arre', 'gue', 'mine', 's', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 2, 4, 4, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je vis à Aix-en-Provence, mais je passe quelques jours à Marseille.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Aix', '-', 'en', '-', 'Provence', ',', '▁mais', '▁je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Marseille', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0],
      dtype=int64)}, {'text': 'Je viens de Tarbes, en visite à Lourdes pour un événement.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Tarbes', ',', '▁en', '▁visite', '▁à', '▁Lourdes', '▁pour', '▁un', '▁événement', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Originaire de Villeneuve-sur-Lot, je suis en déplacement à Agen.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Villeneuve', '-', 'sur', '-', 'L', 'ot', ',', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁A', 'gen', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Saint-Nazaire, actuellement à Nantes pour le travail.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Saint', '-', 'Nazaire', ',', '▁actuellement', '▁à', '▁Nantes', '▁pour', '▁le', '▁travail', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je viens de Tulle, actuellement en visite à Brive.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Tu', 'lle', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Brive', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis de Toulon, mais je passe quelques jours à Fréjus.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Toulon', ',', '▁mais', '▁je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Fréjus', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je vis à Bastia, mais je suis en déplacement à Ajaccio.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Bastia', ',', '▁mais', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁', 'Ajaccio', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Originaire de Perpignan, je suis en visite à Carcassonne.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Perpignan', ',', '▁je', '▁suis', '▁en', '▁visite', '▁à', '▁Carcassonne', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je viens de Montluçon, actuellement à Moulins pour un séminaire.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Mont', 'lu', 'çon', ',', '▁actuellement', '▁à', '▁Moulin', 's', '▁pour', '▁un', '▁séminaire', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Dole, mais je suis en déplacement à Besançon.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Dol', 'e', ',', '▁mais', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Besançon', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je vis à Châlons-en-Champagne, actuellement en déplacement à Reims.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Châ', 'lons', '-', 'en', '-', 'Champ', 'agne', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁Reims', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 2, 0, 0],
      dtype=int64)}, {'text': 'Originaire de Gap, je suis actuellement à Sisteron pour un projet.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁G', 'ap', ',', '▁je', '▁suis', '▁actuellement', '▁à', '▁Si', 'ster', 'on', '▁pour', '▁un', '▁projet', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 2, 4, 4, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je viens de Lens, en déplacement à Arras pour le travail.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Lens', ',', '▁en', '▁déplacement', '▁à', '▁Ar', 'ras', '▁pour', '▁le', '▁travail', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je vis à Sète, mais je suis en déplacement à Béziers.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁S', 'ète', ',', '▁mais', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Béziers', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis de Royan, en visite à La Rochelle pour quelques jours.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Roy', 'an', ',', '▁en', '▁visite', '▁à', '▁La', '▁Rochelle', '▁pour', '▁quelques', '▁jours', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Originaire de Saint-Quentin, actuellement en déplacement à Laon.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Saint', '-', 'Quentin', ',', '▁actuellement', '▁en', '▁déplacement', '▁à', '▁La', 'on', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je vis à Fougères, mais je suis en déplacement à Vitré.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Fou', 'gère', 's', ',', '▁mais', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Vit', 'ré', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je viens de Manosque, en visite à Digne-les-Bains pour le travail.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Man', 'osque', ',', '▁en', '▁visite', '▁à', '▁D', 'igne', '-', 'les', '-', 'Bains', '▁pour', '▁le', '▁travail', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je suis de Dinan, actuellement à Saint-Malo pour une conférence.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Din', 'an', ',', '▁actuellement', '▁à', '▁Saint', '-', 'Malo', '▁pour', '▁une', '▁conférence', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 4, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Originaire de Saint-Jean-de-Luz, actuellement à Bayonne pour le travail.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Saint', '-', 'Jean', '-', 'de', '-', 'L', 'uz', ',', '▁actuellement', '▁à', '▁Bayonne', '▁pour', '▁le', '▁travail', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 2, 0, 0, 0, 0, 0],
      dtype=int64)}, {'text': 'Je vis à Chambéry, mais je passe quelques jours à Aix-les-Bains.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Chambéry', ',', '▁mais', '▁je', '▁passe', '▁quelques', '▁jours', '▁à', '▁Aix', '-', 'les', '-', 'Bains', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 4, 4, 0, 0],
      dtype=int64)}, {'text': 'Je viens de Lannion, en visite à Morlaix pour le weekend.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁La', 'nni', 'on', ',', '▁en', '▁visite', '▁à', '▁Morlaix', '▁pour', '▁le', '▁weekend', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Tulle, mais je suis actuellement en visite à Limoges.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Tu', 'lle', ',', '▁mais', '▁je', '▁suis', '▁actuellement', '▁en', '▁visite', '▁à', '▁Limoges', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Originaire de Morlaix, en déplacement à Brest pour un événement.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Morlaix', ',', '▁en', '▁déplacement', '▁à', '▁Brest', '▁pour', '▁un', '▁événement', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de La Baule, actuellement en visite à Saint-Nazaire.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁La', '▁Bau', 'le', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Saint', '-', 'Nazaire', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je vis à Alès, mais je passe quelques jours à Uzès.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁A', 'lès', ',', '▁mais', '▁je', '▁passe', '▁quelques', '▁jours', '▁à', '▁U', 'z', 'ès', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 0, 0], dtype=int64)}, {'text': 'Je viens de Sisteron, en déplacement à Manosque pour un projet.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Si', 'ster', 'on', ',', '▁en', '▁déplacement', '▁à', '▁Man', 'osque', '▁pour', '▁un', '▁projet', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Dieppe, mais je suis en visite à Rouen pour le travail.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Dieppe', ',', '▁mais', '▁je', '▁suis', '▁en', '▁visite', '▁à', '▁Rouen', '▁pour', '▁le', '▁travail', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Originaire de Pamiers, actuellement à Foix pour une conférence.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Pam', 'iers', ',', '▁actuellement', '▁à', '▁Foi', 'x', '▁pour', '▁une', '▁conférence', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je vis à Dax, mais je suis en déplacement à Mont-de-Marsan.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁D', 'ax', ',', '▁mais', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Mont', '-', 'de', '-', 'Mar', 'san', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 4, 4, 4, 0, 0],
      dtype=int64)}, {'text': 'Je viens de Bergerac, actuellement en visite à Périgueux.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Bergerac', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Périgueux', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je suis de Cosne-Cours-sur-Loire, en déplacement à Nevers.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁Cos', 'ne', '-', 'Cour', 's', '-', 'sur', '-', 'Loire', ',', '▁en', '▁déplacement', '▁à', '▁Never', 's', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 2, 4, 0, 0],
      dtype=int64)}, {'text': 'Originaire de Saint-Lô, actuellement en visite à Coutances.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Saint', '-', 'L', 'ô', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Cout', 'ances', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 3, 3, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je vis à Crest, mais je suis en déplacement à Valence.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁C', 'r', 'est', ',', '▁mais', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁Valence', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], dtype=int64)}, {'text': 'Je viens de Sens, en visite à Auxerre pour un séminaire.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Sens', ',', '▁en', '▁visite', '▁à', '▁', 'Auxerre', '▁pour', '▁un', '▁séminaire', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 4, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je suis de Lure, actuellement à Vesoul pour un projet.', 'tokens': ['<s>', '▁Je', '▁suis', '▁de', '▁L', 'ure', ',', '▁actuellement', '▁à', '▁Ve', 's', 'oul', '▁pour', '▁un', '▁projet', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 2, 4, 4, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Originaire de Douarnenez, en déplacement à Quimper pour le travail.', 'tokens': ['<s>', '▁Originaire', '▁de', '▁Dou', 'arn', 'en', 'ez', ',', '▁en', '▁déplacement', '▁à', '▁Quimper', '▁pour', '▁le', '▁travail', '.', '</s>'], 'predictions': array([0, 0, 0, 1, 3, 3, 3, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0], dtype=int64)}, {'text': 'Je vis à Saintes, mais je suis en déplacement à La Rochelle.', 'tokens': ['<s>', '▁Je', '▁vis', '▁à', '▁Sainte', 's', ',', '▁mais', '▁je', '▁suis', '▁en', '▁déplacement', '▁à', '▁La', '▁Rochelle', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 0, 2, 4, 0, 0], dtype=int64)}, {'text': 'Je viens de Vesoul, actuellement en visite à Luxeuil-les-Bains.', 'tokens': ['<s>', '▁Je', '▁viens', '▁de', '▁Ve', 's', 'oul', ',', '▁actuellement', '▁en', '▁visite', '▁à', '▁Lux', 'euil', '-', 'les', '-', 'Bains', '.', '</s>'], 'predictions': array([0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 0, 0, 2, 4, 4, 4, 4, 4, 0, 0],
      dtype=int64)}]
df_results = pd.DataFrame(results)
df_results.head()


## Distribution des scores de confiance


Ce graphique montre la distribution des scores de confiance pour chaque prédiction :
- **L'axe X** représente les scores de confiance.
- **L'axe Y** représente le nombre de prédictions ayant ce niveau de confiance.


In [ ]:

import matplotlib.pyplot as plt

# Extract confidences
confidences = [np.max(row["predictions"]) for row in results]
plt.figure(figsize=(10, 6))
plt.hist(confidences, bins=10, color='skyblue', alpha=0.7)
plt.title("Distribution des scores de confiance")
plt.xlabel("Score de confiance")
plt.ylabel("Nombre de prédictions")
plt.show()


## Courbe cumulative des scores de confiance


Ce graphique montre la proportion cumulative des prédictions atteignant un certain score de confiance.


In [ ]:

sorted_confidences = sorted(confidences)
cumulative = np.cumsum(sorted_confidences) / sum(sorted_confidences)
plt.figure(figsize=(10, 6))
plt.plot(sorted_confidences, cumulative, color='blue')
plt.title("Courbe cumulative des scores de confiance")
plt.xlabel("Score de confiance")
plt.ylabel("Proportion cumulative")
plt.grid()
plt.show()
